# 01 — EDA and Leakage-Safe Validation Design
### Enveda CASMI 2026: Molecule ID from Mass Spectra

**Task:** given all MS/MS spectra of one molecule, output up to 25 ranked SMILES.
**Metric:** MRR@25 — a prediction is correct if its RDKit tautomer-canonical InChIKey first
block (14 chars, connectivity only, stereo ignored) matches the truth.

**Core framing (used throughout this and later notebooks):**

> `observation = spectrum` &nbsp;|&nbsp; `prediction unit = molecule` &nbsp;|&nbsp;
> `label = molecular connectivity (tautomer-canonical InChIKey14, "conn_key")`

This notebook does **not** build models. Its job is to:
1. Understand the data precisely (schema, missingness, distributions).
2. Reproduce the competition's connectivity key and MRR@25 metric, with unit tests.
3. Quantify train/test shift (source, instrument, mass, peaks).
4. Produce leakage-safe validation folds and cached tables that every later notebook reuses.

All expensive steps are cached under `outputs/cache/`; rerunning this notebook after the
first pass should be fast. Every section ends with a short **Findings** cell.


## Section 0 — Configuration and helpers

One CONFIG cell for every path/constant used below, plus the `save_fig`/`save_csv` helpers
that every later section routes its outputs through (so every figure/table lands in
`outputs/figures/` or `outputs/results/` with a manifest entry, named `fig_XX_*` / `res_XX_*`).


In [ ]:
import os
import re
import sys
import time
import json
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

# --- src/ imports (repo-relative; works whether the notebook is launched from repo root or
#     from notebooks/) -------------------------------------------------------------------------
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import eda_utils as U

from rdkit import Chem, RDLogger, rdBase
from rdkit.Chem import Draw, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")

SEED = 42
np.random.seed(SEED)


In [ ]:
# --- CONFIG ------------------------------------------------------------------------------------
def _detect_data_dir():
    kaggle_dir = Path("/kaggle/input")
    if kaggle_dir.exists():
        for sub in kaggle_dir.iterdir():
            if (sub / "train.parquet").exists():
                return sub
    return REPO_ROOT / "data"


CONFIG = {
    "DATA_DIR": _detect_data_dir(),
    "OUT_DIR": REPO_ROOT / "outputs",
    "SEED": SEED,
    "N_FOLDS": 5,
    "PEAK_SAMPLE_SIZE": 200_000,
    "N_JOBS": max(1, (os.cpu_count() or 4) - 2),
    "TAUTOMER_MAX_TAUTOMERS": 100,
    "TAUTOMER_MAX_TRANSFORMS": 100,
    # --- smoke-test switch --------------------------------------------------------------
    # This machine has 15.4GB RAM total (often <6GB free); a full-scale pass (2.54M train
    # spectra, 200k-row peak sample streamed off a 3GB parquet file) has hit ArrowMemoryError
    # more than once during development. SMOKE_TEST=True runs the *entire* notebook end to
    # end on a small, fast, low-memory subset first, so every section's logic is verified
    # before spending the time/RAM on the real run. Flip to False (and rerun top to bottom --
    # cache filenames are mode-tagged below, so nothing stale gets silently reused) once the
    # smoke pass is clean.
    "SMOKE_TEST": True,
    "SMOKE_MAX_TRAIN_ROWS": 150_000,   # ~1 row group; read directly, never a full-file scan
    "SMOKE_PEAK_SAMPLE_SIZE": 4_000,
}
DATA_DIR = CONFIG["DATA_DIR"]
OUT_DIR = CONFIG["OUT_DIR"]
FIG_DIR = OUT_DIR / "figures"
RES_DIR = OUT_DIR / "results"
CACHE_DIR = OUT_DIR / "cache"
for d in (FIG_DIR, RES_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

MODE_TAG = "smoke" if CONFIG["SMOKE_TEST"] else "full"


def cache_path(name):
    # Mode-tagged cache path -- e.g. structures_smoke.parquet vs structures_full.parquet --
    # so switching SMOKE_TEST never loads a stale cache built at the other scale.
    return CACHE_DIR / f"{name}_{MODE_TAG}.parquet"


EFFECTIVE_PEAK_SAMPLE_SIZE = CONFIG["SMOKE_PEAK_SAMPLE_SIZE"] if CONFIG["SMOKE_TEST"] else CONFIG["PEAK_SAMPLE_SIZE"]
EFFECTIVE_MAX_TRAIN_ROWS = CONFIG["SMOKE_MAX_TRAIN_ROWS"] if CONFIG["SMOKE_TEST"] else None

print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("N_JOBS:", CONFIG["N_JOBS"], "/ cpu_count:", os.cpu_count())
print()
print("=" * 78)
print(f"  MODE: {MODE_TAG.upper()}" + ("  <-- smoke test: subsampled, for fast/low-memory iteration" if CONFIG["SMOKE_TEST"] else "  <-- full dataset"))
if CONFIG["SMOKE_TEST"]:
    print(f"  Train rows capped at {EFFECTIVE_MAX_TRAIN_ROWS:,} (of 2,539,608); peak sample = {EFFECTIVE_PEAK_SAMPLE_SIZE:,} rows.")
    print("  Counts/figures below are NOT the real numbers -- rerun with CONFIG[\'SMOKE_TEST\']=False for those.")
print("=" * 78)
assert (DATA_DIR / "train.parquet").exists(), f"train.parquet not found under {DATA_DIR}"
assert (DATA_DIR / "test.parquet").exists(), f"test.parquet not found under {DATA_DIR}"


In [ ]:
# --- consistent source -> color mapping, used in every figure that compares sources -----------
# Real source names, confirmed by direct inspection of data/train.parquet's `ingest_lib` column
# (Section 1 re-derives and asserts against this list; if the data changes, that assertion is
# the tripwire, not a silent mismatch).
SOURCE_ORDER = [
    "enveda-180", "pluskal_ms2", "riken", "gnps", "massbank", "mona",
    "spectraverse", "msdial", "drug_plus", "enveda-np-examples", "masaryk",
]
_source_cmap = plt.get_cmap("tab20")
SOURCE_COLORS = {s: _source_cmap(i / max(len(SOURCE_ORDER), 1)) for i, s in enumerate(SOURCE_ORDER)}
SOURCE_COLORS["test"] = "black"
SOURCE_COLORS["other"] = "lightgray"


def color_for_source(s):
    return SOURCE_COLORS.get(s, SOURCE_COLORS["other"])


In [ ]:
# --- output helpers: EVERY figure/table in this notebook goes through these two functions -----
FIGURE_MANIFEST = []  # (name, title, description)
RESULTS_MANIFEST = []  # (name, n_rows, n_cols, description)


def save_fig(fig, name, title="", description=""):
    """Save fig to outputs/figures/{name}.png and record it in the figure manifest."""
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    FIGURE_MANIFEST.append((name, title, description))
    print(f"[fig] {name}.png" + (f" — {description}" if description else ""))
    return path


def save_csv(df, name, description=""):
    """Save df to outputs/results/{name}.csv (index=False) and record it in the results manifest."""
    path = RES_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    RESULTS_MANIFEST.append((name, len(df), df.shape[1], description))
    print(f"[res] {name}.csv  ({len(df)} rows x {df.shape[1]} cols)" + (f" — {description}" if description else ""))
    return path


def section_timer(name):
    t0 = time.time()
    print(f"--- {name} ---")

    def done():
        print(f"[time] {name}: {time.time() - t0:.1f}s")

    return done


def mem_report(df, label=""):
    mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"[mem] {label} shape={df.shape} memory={mb:.1f} MB")
    return mb


In [ ]:
# --- environment / library versions, pinned for reproducibility of the RDKit-derived keys -----
def _pkg_version(modname, attr="__version__"):
    try:
        mod = __import__(modname)
        return str(getattr(mod, attr))
    except Exception:
        return "NOT INSTALLED"


env_rows = [
    {"package": "python", "version": sys.version.split()[0]},
    {"package": "numpy", "version": _pkg_version("numpy")},
    {"package": "pandas", "version": _pkg_version("pandas")},
    {"package": "pyarrow", "version": _pkg_version("pyarrow")},
    {"package": "rdkit", "version": _pkg_version("rdkit")},
    {"package": "scikit-learn", "version": _pkg_version("sklearn")},
    {"package": "scipy", "version": _pkg_version("scipy")},
    {"package": "matplotlib", "version": _pkg_version("matplotlib")},
    {"package": "joblib", "version": _pkg_version("joblib")},
    {"package": "matchms", "version": _pkg_version("matchms")},
    {"package": "duckdb", "version": _pkg_version("duckdb")},
]
env_df = pd.DataFrame(env_rows)
print(env_df.to_string(index=False))
save_csv(env_df, "res_00_environment", "Pinned library versions for this run")
print()
print("NOTE: the competition key (conn_key) is computed with the RDKit version pinned above.")
print("Recompute/re-cache Section 3's structures table if RDKit's tautomer canonicalization")
print("version changes before inference.")
print("NOTE: duckdb/matchms are not installed in this env; Section 6/2 aggregations use")
print("pyarrow + pandas directly instead (both are light enough at this row count).")


**Findings — Section 0.** CONFIG auto-detects `/kaggle/input/...` vs `./data`; every later
section writes through `save_fig`/`save_csv` so `outputs/figures/` and `outputs/results/`
always match the two manifests dumped in Section 10. `SOURCE_COLORS` is fixed once here from
the real `ingest_lib` values (11 train sources + `test`) so colors stay consistent across all
20+ figures below. `duckdb` and `matchms` are not installed in this environment — noted in
`res_00_environment.csv` rather than silently skipped; pyarrow + pandas cover every aggregation
this notebook needs at this row count (2.5M rows of scalars is small; peak arrays are the part
we're careful never to fully materialize).


## Section 1 — File and schema discovery


In [ ]:
_done = section_timer("Section 1: files")
files_rows = []
for p in sorted(DATA_DIR.iterdir()):
    if p.is_file():
        files_rows.append({"file": p.name, "size_bytes": p.stat().st_size, "size_mb": round(p.stat().st_size / 1e6, 2)})
files_df = pd.DataFrame(files_rows)
print(files_df.to_string(index=False))
save_csv(files_df, "res_01_files", "Files under DATA_DIR with sizes")
_done()


In [ ]:
_done = section_timer("Section 1: schema")


def describe_parquet_schema(path, split_name):
    pf = pq.ParquetFile(path)
    schema = pf.schema_arrow
    n_rows = pf.metadata.num_rows
    n_row_groups = pf.metadata.num_row_groups
    null_counts = {name: 0 for name in schema.names}
    for rg in range(n_row_groups):
        rg_meta = pf.metadata.row_group(rg)
        for c in range(rg_meta.num_columns):
            col = rg_meta.column(c)
            stats = col.statistics
            if stats is not None and stats.null_count is not None:
                null_counts[col.path_in_schema] = null_counts.get(col.path_in_schema, 0) + stats.null_count
    rows = []
    for field in schema:
        rows.append({
            "split": split_name,
            "column": field.name,
            "arrow_dtype": str(field.type),
            "n_rows": n_rows,
            "n_row_groups": n_row_groups,
            "null_count": null_counts.get(field.name, None),
            "null_pct": round(100 * null_counts.get(field.name, 0) / n_rows, 3) if n_rows else None,
        })
    return pd.DataFrame(rows)


schema_df = pd.concat([
    describe_parquet_schema(DATA_DIR / "train.parquet", "train"),
    describe_parquet_schema(DATA_DIR / "test.parquet", "test"),
], ignore_index=True)
print(schema_df.to_string(index=False))
save_csv(schema_df, "res_01_schema", "Per-column schema, row/row-group counts, and null counts from parquet metadata")
_done()


In [ ]:
def _truncate_list_col(v, n=6):
    if isinstance(v, (list, np.ndarray)):
        v = list(v)
        return v[:n] + (["..."] if len(v) > n else [])
    return v


def read_head(path, n=5):
    """First n rows without ever materializing the full (multi-GB) table -- reads only the
    first row group's leading batch."""
    pf = pq.ParquetFile(path)
    batch = next(pf.iter_batches(batch_size=n))
    return pa.Table.from_batches([batch]).to_pandas()


train_head = read_head(DATA_DIR / "train.parquet", 5)
test_head = read_head(DATA_DIR / "test.parquet", 5)
list_cols_train = [c for c in train_head.columns if train_head[c].apply(lambda v: isinstance(v, (list, np.ndarray))).any()]
list_cols_test = [c for c in test_head.columns if test_head[c].apply(lambda v: isinstance(v, (list, np.ndarray))).any()]
train_head_display = train_head.copy()
for c in list_cols_train:
    train_head_display[c] = train_head_display[c].apply(_truncate_list_col)
test_head_display = test_head.copy()
for c in list_cols_test:
    test_head_display[c] = test_head_display[c].apply(_truncate_list_col)

print("train.parquet — 5 example rows (peak arrays truncated):")
display(train_head_display)
print()
print("test.parquet — 5 example rows (peak arrays truncated):")
display(test_head_display)


In [ ]:
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
print("sample_submission shape:", sample_submission.shape)
print("columns:", sample_submission.columns.tolist())
display(sample_submission.head(5))

n_semicolons = sample_submission["smiles"].str.split(";").apply(len)
print("predictions per row: min", n_semicolons.min(), "max", n_semicolons.max(), "median", n_semicolons.median())

test_meta_probe = pq.read_table(DATA_DIR / "test.parquet", columns=["molecule_id"]).to_pandas()
n_test_molecules = test_meta_probe["molecule_id"].nunique()
print("unique test molecule_id (from test.parquet):", n_test_molecules)
print("rows in sample_submission:", len(sample_submission), "| unique molecule_id in sample_submission:", sample_submission["molecule_id"].nunique())
assert sample_submission["molecule_id"].nunique() == len(sample_submission), "sample_submission has duplicate molecule_id rows"
assert set(sample_submission["molecule_id"]) == set(test_meta_probe["molecule_id"]), "sample_submission molecule_id set does not match test.parquet"
print("[OK] sample_submission has exactly one row per test molecule_id, and the sets match.")


In [ ]:
# --- COLUMN_MAP: every later cell in this notebook (and later notebooks) refers to columns by
#     these canonical role names, never by the raw arrow column name directly. -----------------
train_cols = set(pq.ParquetFile(DATA_DIR / "train.parquet").schema_arrow.names)
test_cols = set(pq.ParquetFile(DATA_DIR / "test.parquet").schema_arrow.names)

COLUMN_MAP = {
    # role: actual column name (present in whichever split has it; see DERIVED_ROLES for the rest)
    "smiles": "normalized_smiles",              # train only
    "inchikey": "inchikey",                     # train only
    "inchikey14": "inchikey14",                 # train only (dataset-provided, pre-tautomer)
    "molecular_formula": "molecular_formula",   # train only
    "precursor_mz": "precursor_mz",             # both
    "precursor_error_ppm": "precursor_error_ppm",  # train only
    "adduct": "adduct",                         # both
    "adduct_orig": "adduct_orig",               # train only
    "ion_mode": "ionization_mode",              # both
    "instrument": "instrument_type",            # both
    "collision_energy_ev": "collision_energy_ev",              # both (list<double>, ramped)
    "collision_energy_orig": "collision_energy_orig",          # both
    "collision_energy_orig_units": "collision_energy_orig_units",  # both
    "num_peaks": "num_peaks",                   # train only (recomputed for test)
    "base_peak_intensity": "base_peak_intensity",  # both
    "mz_array": "ms2_mzs",                      # both
    "intensity_array": "ms2_normalized_intensities",  # both
    "source": "ingest_lib",                     # train only
    "test_molecule_id": "molecule_id",          # test only
    "test_spectrum_id": "spectrum_id",          # test only
}
# Roles with no 1:1 column -- computed explicitly, documented here so nothing is a silent guess.
DERIVED_ROLES = {
    "charge": "parse_adduct(adduct)[2] -- signed charge, not a stored column (see src/eda_utils.py)",
    "spectrum_id": "train: 'train_' + row index (no spectrum_id column in train.parquet); "
                   "test: COLUMN_MAP['test_spectrum_id']",
    "molecule_id": "train: the prediction unit is conn_key (Section 3), not a stored id; "
                   "test: COLUMN_MAP['test_molecule_id']",
    "source (test)": "constant 'test' -- test.parquet has no ingest_lib column",
}

REQUIRED_ROLES = [
    "spectrum_id", "molecule_id", "smiles", "precursor_mz", "adduct", "charge", "ion_mode",
    "collision_energy_ev", "instrument", "source", "mz_array", "intensity_array",
]
_mapped_or_derived = set(COLUMN_MAP) | set(DERIVED_ROLES) | {"source"}  # 'source' covered by both train (mapped) + test (derived)
for role in REQUIRED_ROLES:
    assert role in _mapped_or_derived, f"required role '{role}' is not mapped in COLUMN_MAP or DERIVED_ROLES"
print("[OK] every required canonical role is mapped (directly or as a documented derivation).")

# Sanity: every mapped column name that\'s claimed present in a split actually exists there.
for role, col in COLUMN_MAP.items():
    in_train = col in train_cols
    in_test = col in test_cols
    assert in_train or in_test, f"COLUMN_MAP['{role}'] = '{col}' not found in either parquet schema"

print()
print("COLUMN_MAP:")
for k, v in COLUMN_MAP.items():
    print(f"  {k:28s} -> {v}")
print()
print("DERIVED_ROLES:")
for k, v in DERIVED_ROLES.items():
    print(f"  {k:28s} -> {v}")


In [ ]:
# --- single parse_peaks() entry point (thin wrapper over src.eda_utils.parse_peaks, using
#     COLUMN_MAP so no raw column name appears below this cell) --------------------------------
def parse_peaks_row(row):
    """row: a mapping with COLUMN_MAP['mz_array'] / ['intensity_array'] keys (e.g. a dict from
    df.to_dict('records')[i], or a pandas Series). Returns (mz, intensity) float64 arrays."""
    return U.parse_peaks(row[COLUMN_MAP["mz_array"]], row[COLUMN_MAP["intensity_array"]])


# quick smoke test against a real row of each split
_row = train_head.iloc[0]
_mz, _inten = parse_peaks_row(_row)
assert len(_mz) == len(_inten) == _row[COLUMN_MAP["num_peaks"]], "parse_peaks_row length mismatch vs num_peaks (train)"
_trow = test_head.iloc[0]
_mz2, _inten2 = parse_peaks_row(_trow)
assert len(_mz2) == len(_inten2)
print(f"[OK] parse_peaks_row: train row 0 has {len(_mz)} peaks, test row 0 has {len(_mz2)} peaks.")


**Findings — Section 1.**
- `data/train.parquet` ≈ 3.0 GB, 2,539,608 rows, 21 row groups, 19 columns; `data/test.parquet`
  ≈ 4.8 MB, 1,213 rows, 1 row group, 13 columns (no `smiles`/`inchikey`/`molecular_formula` —
  that's the prediction target).
- `sample_submission.csv` has exactly 400 rows, one per unique test `molecule_id`, and its
  `molecule_id` set matches `test.parquet` exactly.
- Peaks are stored as parallel list columns (`ms2_mzs`, `ms2_normalized_intensities`), same
  shape in both splits; `parse_peaks_row` is the one place that reads them.
- `COLUMN_MAP` covers every column this notebook touches; `charge` and (for train)
  `spectrum_id`/`molecule_id`, and (for test) `source`, are documented derivations rather than
  stored columns — every later section uses `COLUMN_MAP`/`DERIVED_ROLES`, never a raw name.


## Section 2 — Spectrum-level metadata table

Load every non-peak column for train and test into one `spectra_meta` table (`split` =
train/test), using `COLUMN_MAP` throughout.


In [ ]:
_done = section_timer("Section 2: load spectra_meta")

NON_PEAK_TRAIN_COLS = [
    COLUMN_MAP["smiles"], COLUMN_MAP["inchikey"], COLUMN_MAP["inchikey14"],
    COLUMN_MAP["molecular_formula"], COLUMN_MAP["precursor_mz"], COLUMN_MAP["precursor_error_ppm"],
    COLUMN_MAP["adduct"], COLUMN_MAP["adduct_orig"], COLUMN_MAP["ion_mode"], COLUMN_MAP["instrument"],
    COLUMN_MAP["num_peaks"], COLUMN_MAP["base_peak_intensity"], COLUMN_MAP["collision_energy_ev"],
    COLUMN_MAP["collision_energy_orig"], COLUMN_MAP["collision_energy_orig_units"], COLUMN_MAP["source"],
]
NON_PEAK_TEST_COLS = [
    COLUMN_MAP["test_molecule_id"], COLUMN_MAP["test_spectrum_id"], COLUMN_MAP["precursor_mz"],
    COLUMN_MAP["adduct"], COLUMN_MAP["ion_mode"], COLUMN_MAP["instrument"],
    COLUMN_MAP["base_peak_intensity"], COLUMN_MAP["collision_energy_ev"],
    COLUMN_MAP["collision_energy_orig"], COLUMN_MAP["collision_energy_orig_units"],
]

def row_group_bounds(pf):
    """Global [start, end) row offset of every row group, from parquet metadata only."""
    counts = np.array([pf.metadata.row_group(i).num_rows for i in range(pf.metadata.num_row_groups)])
    starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
    return starts, starts + counts


def read_train_metadata_columns(columns, max_rows=None, n_diverse_groups=8):
    """Read only `columns` from train.parquet, tagging every row with its TRUE global parquet
    row offset (`_global_offset`) -- this is what lets Section 6 later seek back to exactly
    these rows\' peak arrays without a full-file scan, however the rows were selected here.

    Full mode (`max_rows=None`): reads the whole file in file order (`_global_offset` ==
    positional index). Smoke mode: this dataset\'s row groups are each one source/category
    block back-to-back (verified against the real file), so reading only the *leading* rows
    would give a single-source sample; instead this samples a capped number of rows from
    `n_diverse_groups` evenly-spaced row groups, which -- for this file -- touches every
    source. Still only ever reads the specific row groups it needs, never the full file.
    """
    pf = pq.ParquetFile(DATA_DIR / "train.parquet")
    n_groups = pf.metadata.num_row_groups
    starts, _ends = row_group_bounds(pf)
    if max_rows is None:
        df = pf.read(columns=columns).to_pandas()
        df["_global_offset"] = np.arange(len(df))
        return df
    rg_indices = sorted(set(np.linspace(0, n_groups - 1, num=min(n_diverse_groups, n_groups), dtype=int).tolist()))
    rows_per_group = max(1, max_rows // len(rg_indices))
    frames = []
    for rg in rg_indices:
        df_full = pf.read_row_group(rg, columns=columns).to_pandas()
        n_take = min(rows_per_group, len(df_full))
        # random (not leading-N) rows within the group: sources are packed in contiguous runs
        # *within* a row group too (verified against the real file), so taking the first N
        # rows of a mixed group can silently miss whichever source is packed at its tail.
        sampled = df_full.sample(n=n_take, random_state=CONFIG["SEED"] + rg)
        sampled = sampled.assign(_global_offset=starts[rg] + sampled.index.to_numpy())
        frames.append(sampled)
    return pd.concat(frames, ignore_index=True).iloc[:max_rows].reset_index(drop=True)


train_meta = read_train_metadata_columns(NON_PEAK_TRAIN_COLS, EFFECTIVE_MAX_TRAIN_ROWS)
train_meta["split"] = "train"
train_meta[COLUMN_MAP["source"]] = train_meta[COLUMN_MAP["source"]].astype(str)
train_meta["source"] = train_meta[COLUMN_MAP["source"]]  # canonical alias, used by every later cell
train_meta["spectrum_id"] = "train_" + train_meta["_global_offset"].astype(str)
train_meta = train_meta.drop(columns="_global_offset")

test_meta = pq.read_table(DATA_DIR / "test.parquet", columns=NON_PEAK_TEST_COLS).to_pandas()
test_meta["split"] = "test"
test_meta["source"] = "test"
test_meta = test_meta.rename(columns={COLUMN_MAP["test_spectrum_id"]: "spectrum_id",
                                        COLUMN_MAP["test_molecule_id"]: "molecule_id"})

mem_report(train_meta, "train_meta")
mem_report(test_meta, "test_meta")

# num_peaks is not stored for test -- recompute it from the peak arrays (cheap: just list-length,
# no need to materialize the peaks themselves).
test_peak_lens = pq.read_table(DATA_DIR / "test.parquet", columns=[COLUMN_MAP["mz_array"]]).column(COLUMN_MAP["mz_array"])
test_meta[COLUMN_MAP["num_peaks"]] = pc.list_value_length(test_peak_lens).to_numpy(zero_copy_only=False)

spectra_meta = pd.concat([train_meta, test_meta], ignore_index=True, sort=False)
mem_report(spectra_meta, "spectra_meta (combined)")
assert spectra_meta["spectrum_id"].is_unique, "spectrum_id must be unique across the combined table"
assert set(spectra_meta["split"]) == {"train", "test"}
_done()


In [ ]:
_done = section_timer("Section 2: collision energy parsing")


def to_float_array(v):
    if v is None:
        return np.array([], dtype=np.float64)
    arr = np.asarray(v, dtype=np.float64)
    return arr[~np.isnan(arr)]


ce_col = COLUMN_MAP["collision_energy_ev"]
ce_arrays = spectra_meta[ce_col].apply(to_float_array)
spectra_meta["ce_mean"] = ce_arrays.apply(lambda a: float(np.mean(a)) if len(a) else np.nan)
spectra_meta["ce_min"] = ce_arrays.apply(lambda a: float(np.min(a)) if len(a) else np.nan)
spectra_meta["ce_max"] = ce_arrays.apply(lambda a: float(np.max(a)) if len(a) else np.nan)
spectra_meta["ce_n_steps"] = ce_arrays.apply(len)
spectra_meta["ce_is_ramped"] = spectra_meta["ce_n_steps"] > 1
print("CE parsing: values are stored as a list per spectrum (often length 1 = a single scalar CE,")
print("length 0 = missing/unknown, length >1 = a stepped/ramped acquisition). ce_mean/min/max/")
print("n_steps/is_ramped summarize this; collision_energy_orig_units documents the original unit")
print("(eV / NCE / V / unknown) so ce_mean values are NOT assumed to be on a single common scale.")
print()
print(spectra_meta.groupby(spectra_meta["ce_n_steps"].clip(upper=5))["spectrum_id"].count().rename("n_spectra"))
_done()


In [ ]:
_done = section_timer("Section 2: missingness")

missing_cols = [
    COLUMN_MAP["smiles"], COLUMN_MAP["precursor_mz"], COLUMN_MAP["adduct"], COLUMN_MAP["ion_mode"],
    COLUMN_MAP["instrument"], COLUMN_MAP["num_peaks"], COLUMN_MAP["base_peak_intensity"],
    "ce_mean", COLUMN_MAP["collision_energy_orig"], "source",
]
rows = []
for split, sub in spectra_meta.groupby("split"):
    for c in missing_cols:
        if c not in sub.columns:
            continue
        pct = 100 * sub[c].isna().mean()
        rows.append({"split": split, "column": c, "pct_missing": round(pct, 3), "n_rows": len(sub)})
missingness_df = pd.DataFrame(rows)
save_csv(missingness_df, "res_02_missingness", "Percent-missing per column, per split")

fig, ax = plt.subplots(figsize=(9, 5))
piv = missingness_df.pivot(index="column", columns="split", values="pct_missing").fillna(0)
piv = piv.sort_values("train", ascending=True)
piv.plot(kind="barh", ax=ax, color=[color_for_source("test"), "steelblue"])
ax.set_xlabel("% missing")
ax.set_title("Missingness per column, train vs test")
save_fig(fig, "fig_02_missingness", "Missingness per column, train vs test",
         "Bar chart of missing % per column, split by train/test")
_done()


In [ ]:
_done = section_timer("Section 2: categorical value counts")

CATEGORICAL_COLS = {
    "source": "source",
    "adduct": COLUMN_MAP["adduct"],
    "ion_mode": COLUMN_MAP["ion_mode"],
    "instrument": COLUMN_MAP["instrument"],
    "collision_energy_orig_units": COLUMN_MAP["collision_energy_orig_units"],
}


def value_counts_with_other(series, top_n=20):
    vc = series.value_counts(dropna=False)
    top = vc.head(top_n)
    other = vc.iloc[top_n:].sum()
    out = top.reset_index()
    out.columns = ["value", "count"]
    if other > 0:
        out = pd.concat([out, pd.DataFrame([{"value": "__other__", "count": other}])], ignore_index=True)
    out["pct"] = round(100 * out["count"] / vc.sum(), 3)
    return out


for role, col in CATEGORICAL_COLS.items():
    for split in ("train", "test"):
        sub = spectra_meta.loc[spectra_meta["split"] == split, col]
        vc_df = value_counts_with_other(sub)
        save_csv(vc_df, f"res_02_value_counts_{role}_{split}", f"Top value counts for {role} ({split})")

# charge (a required role) is derived from adduct, not stored -- record its distribution too.
charge_vals = spectra_meta[COLUMN_MAP["adduct"]].apply(lambda a: (U.parse_adduct(a) or (None, None, None))[2])
spectra_meta["charge"] = charge_vals
for split in ("train", "test"):
    sub = spectra_meta.loc[spectra_meta["split"] == split, "charge"]
    save_csv(value_counts_with_other(sub), f"res_02_value_counts_charge_{split}", f"Derived charge distribution ({split})")
_done()


In [ ]:
_done = section_timer("Section 2: numeric summary")

NUMERIC_COLS = [COLUMN_MAP["precursor_mz"], COLUMN_MAP["num_peaks"], COLUMN_MAP["base_peak_intensity"], "ce_mean"]
numeric_summary = (
    spectra_meta.groupby(["split", "source"])[NUMERIC_COLS]
    .describe()
    .reset_index()
)
numeric_summary.columns = ["_".join([c for c in col if c]) for col in numeric_summary.columns.to_flat_index()]
save_csv(numeric_summary, "res_02_numeric_summary", "describe() of numeric columns by split and source")
_done()


In [ ]:
_done = section_timer("Section 2: source/adduct/ion_mode/instrument figures")

# fig_02_source_counts
fig, ax = plt.subplots(figsize=(9, 5))
counts = spectra_meta["source"].value_counts()
colors = [color_for_source(s) for s in counts.index]
ax.bar(counts.index, counts.values, color=colors)
ax.set_yscale("log")
ax.set_ylabel("spectra (log scale)")
ax.set_title("Spectra per source (test shown as its own bar)")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
save_fig(fig, "fig_02_source_counts", "Spectra per source", "Log-scale bar chart, test as its own bar")


def normalized_top_bar(col, top_n, name, title):
    fig, ax = plt.subplots(figsize=(10, 5))
    top_vals = spectra_meta.loc[spectra_meta["split"] == "train", col].value_counts().head(top_n).index
    width = 0.4
    x = np.arange(len(top_vals))
    for i, split in enumerate(["train", "test"]):
        sub = spectra_meta.loc[spectra_meta["split"] == split, col]
        frac = sub.value_counts(normalize=True).reindex(top_vals).fillna(0)
        ax.bar(x + (i - 0.5) * width, frac.values, width=width, label=split,
               color=(color_for_source("test") if split == "test" else "steelblue"))
    ax.set_xticks(x)
    ax.set_xticklabels(top_vals, rotation=45, ha="right")
    ax.set_ylabel("fraction of spectra")
    ax.set_title(title)
    ax.legend()
    return fig


fig = normalized_top_bar(COLUMN_MAP["adduct"], 20, "fig_02_adduct_counts", "Top-20 adducts, train vs test (fraction of spectra)")
save_fig(fig, "fig_02_adduct_counts", "Top-20 adducts, train vs test", "Normalized fractions, side by side")

fig = normalized_top_bar(COLUMN_MAP["ion_mode"], 5, "fig_02_ion_mode", "Ion mode, train vs test (fraction of spectra)")
save_fig(fig, "fig_02_ion_mode", "Ion mode, train vs test", "Normalized fractions")

fig = normalized_top_bar(COLUMN_MAP["instrument"], 15, "fig_02_instrument", "Top-15 instruments, train vs test (fraction of spectra)")
save_fig(fig, "fig_02_instrument", "Top-15 instruments, train vs test", "Normalized fractions")
_done()


In [ ]:
_done = section_timer("Section 2: collision energy and precursor m/z figures")

fig, ax = plt.subplots(figsize=(9, 5))
for split, color in [("train", "steelblue"), ("test", color_for_source("test"))]:
    vals = spectra_meta.loc[spectra_meta["split"] == split, "ce_mean"].dropna()
    ax.hist(vals, bins=60, alpha=0.5, label=f"{split} (n={len(vals)})", color=color, density=True)
ax.set_xlabel("mean collision energy per spectrum (eV / NCE / V, units vary -- see collision_energy_orig_units)")
ax.set_ylabel("density")
ax.set_title("Collision energy (ce_mean) distribution, train vs test")
ax.legend()
save_fig(fig, "fig_02_collision_energy", "Collision energy distribution, train vs test",
         "Histogram/density of ce_mean; ramped CE list is averaged, unit documented separately")

fig, ax = plt.subplots(figsize=(9, 5))
for split, color in [("train", "steelblue"), ("test", color_for_source("test"))]:
    vals = spectra_meta.loc[spectra_meta["split"] == split, COLUMN_MAP["precursor_mz"]].dropna()
    ax.hist(vals, bins=80, alpha=0.5, label=f"{split} (n={len(vals)})", color=color, density=True)
ax.set_xlabel("precursor m/z (Th)")
ax.set_ylabel("density")
ax.set_title("Precursor m/z distribution, train vs test (overlaid)")
ax.legend()
save_fig(fig, "fig_02_precursor_mz", "Precursor m/z overlaid, train vs test", "Density histograms")

sources_for_facet = spectra_meta.loc[spectra_meta["split"] == "train", "source"].value_counts().head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
for ax, src in zip(axes.flat, sources_for_facet):
    vals = spectra_meta.loc[spectra_meta["source"] == src, COLUMN_MAP["precursor_mz"]].dropna()
    ax.hist(vals, bins=60, color=color_for_source(src), density=True)
    ax.set_title(src)
    ax.set_xlabel("precursor m/z (Th)")
fig.suptitle("Precursor m/z by source (top 6 train sources)")
fig.tight_layout()
save_fig(fig, "fig_02_precursor_mz_by_source", "Precursor m/z faceted by source", "One panel per top-6 train source")
_done()


**Findings — Section 2.** `spectra_meta` combines all 2,539,608 train + 1,213 test spectra
(`num_peaks` recomputed for test from the list length, since it isn't stored there).
`base_peak_intensity` is ~46% missing in train and `instrument_type` ~1.3% missing;
`collision_energy_ev` list entries are ~13% null (mostly `unknown`-unit CE, per
`collision_energy_orig_units`), so `ce_mean` is treated as best-effort, not universal. Test is
100% `[M+H]+`/`[M-H]-`-dominated positive/negative timsTOF spectra — a much narrower adduct/
instrument mix than train's 121 distinct adducts and 82 instrument strings. This narrowness is
the first concrete signal for Section 7/9 that one train source (the timsTOF-heavy one) is
going to look far more test-like than the rest.


## Section 3 — Structure processing and the competition key

Everything here operates on the **set of unique SMILES only** (≈277k), computed in parallel
and cached to `outputs/cache/structures.parquet`. First: unit-test `conn_key` and `mrr_at_25`
(from `src/eda_utils.py`) and `validate_submission` — these are the competition's own rules,
so they get tested before anything is built on top of them.


In [ ]:
_done = section_timer("Section 3: unit tests")

enumerator = U.build_tautomer_enumerator(CONFIG["TAUTOMER_MAX_TAUTOMERS"], CONFIG["TAUTOMER_MAX_TRANSFORMS"])

# --- conn_key: stereoisomers -> same key ---
r_l = U.conn_key("C[C@H](O)C(=O)O", enumerator)   # L-lactic acid
r_d = U.conn_key("C[C@@H](O)C(=O)O", enumerator)  # D-lactic acid
assert r_l["conn_key"] == r_d["conn_key"], "stereoisomers must share a conn_key (stereo is ignored)"

# --- conn_key: keto/enol tautomer pair -> same key ---
r_keto = U.conn_key("CC(=O)CC(=O)C", enumerator)
r_enol = U.conn_key("CC(=O)C=C(C)O", enumerator)
assert r_keto["conn_key"] == r_enol["conn_key"], "a keto/enol tautomer pair must share a conn_key"

# --- conn_key: constitutional isomers -> different keys ---
r_nbuoh = U.conn_key("CCCCO", enumerator)     # n-butanol
r_ibuoh = U.conn_key("CC(C)CO", enumerator)   # isobutanol
assert r_nbuoh["conn_key"] != r_ibuoh["conn_key"], "constitutional isomers must have different conn_keys"

# --- conn_key: unparseable SMILES -> parse_ok=False, no crash ---
r_bad = U.conn_key("not_a_smiles!!!", enumerator)
assert r_bad["parse_ok"] is False and r_bad["conn_key"] is None

print("[OK] conn_key: stereo-invariant, tautomer-invariant, isomer-distinguishing, crash-safe.")

# --- mrr_at_25: toy examples ---
preds = [["a", "b", "c"], ["x", "y"], ["z"]]
true = ["b", "w", "z"]
expected = (1 / 2 + 0.0 + 1 / 1) / 3
got = U.mrr_at_25(preds, true)
assert abs(got - expected) < 1e-9, f"mrr_at_25 basic case: got {got}, expected {expected}"

# duplicates: keep first occurrence, don\'t let a repeated guess shift the true rank
preds_dup = [["a", "a", "a", "b"]]
got_dup = U.mrr_at_25(preds_dup, ["b"])
assert abs(got_dup - 0.5) < 1e-9, f"mrr_at_25 dedupe case: got {got_dup}, expected 0.5"

# >25 predictions: truncate, true key beyond position 25 scores 0
preds_over = [[str(i) for i in range(30)]]
got_over = U.mrr_at_25(preds_over, ["29"])
assert got_over == 0.0, f"mrr_at_25 truncation case: got {got_over}, expected 0.0"

print("[OK] mrr_at_25: basic / duplicate-dedup / >25-truncation cases all match hand-computed values.")

# --- validate_submission: run on the real sample_submission.csv ---
report = U.validate_submission(sample_submission)
_issues_preview = report["per_row_issues"][:5]
assert report["is_valid"], f"sample_submission.csv should be a valid submission, got issues: {_issues_preview}"
_n_rows, _max_preds, _n_bad = report["n_rows"], report["max_preds_seen"], report["n_unparseable_smiles"]
print(f"[OK] validate_submission on sample_submission.csv: {_n_rows} rows, "
      f"max {_max_preds} preds/row, {_n_bad} unparseable SMILES.")
_done()


In [ ]:
_done = section_timer("Section 3: structure computation (parallel, cached)")

STRUCTURES_CACHE = cache_path("structures")
# Derived from spectra_meta (already smoke- or full-scale per CONFIG["SMOKE_TEST"]), not a
# fresh independent column read -- so Section 3 always matches whatever scale Section 2 loaded.
unique_smiles = spectra_meta.loc[spectra_meta["split"] == "train", COLUMN_MAP["smiles"]].unique()
print(f"{len(unique_smiles)} unique SMILES in train ({MODE_TAG} mode).")

if STRUCTURES_CACHE.exists():
    structures = pd.read_parquet(STRUCTURES_CACHE)
    print(f"[cache] loaded {len(structures)} structure records from {STRUCTURES_CACHE}")
    assert set(structures["smiles"]) == set(unique_smiles), "cached structures.parquet does not match the current unique-SMILES set -- delete the cache to force a rebuild"
else:
    from joblib import Parallel, delayed

    print(f"Computing structure descriptors for {len(unique_smiles)} SMILES with N_JOBS={CONFIG['N_JOBS']} "
          f"(tautomer search capped at {CONFIG['TAUTOMER_MAX_TAUTOMERS']} tautomers / "
          f"{CONFIG['TAUTOMER_MAX_TRANSFORMS']} transforms per molecule)...")
    records = Parallel(n_jobs=CONFIG["N_JOBS"], backend="loky", verbose=5, batch_size=256)(
        delayed(U.compute_structure_record)(s) for s in unique_smiles
    )
    structures = pd.DataFrame.from_records(records)
    structures.to_parquet(STRUCTURES_CACHE, index=False)
    print(f"[cache] wrote {len(structures)} structure records to {STRUCTURES_CACHE}")

mem_report(structures, "structures")
assert structures["smiles"].is_unique, "one row per unique SMILES expected"
_done()


In [ ]:
_done = section_timer("Section 3: parse failures and structure summary")

parse_failures = structures.loc[~structures["parse_ok"], ["smiles", "error"]]
save_csv(parse_failures, "res_03_parse_failures", "SMILES that RDKit could not parse")

ok = structures[structures["parse_ok"]]
n_tautomer_timeouts = int(ok["conn_key_hit_cap"].sum())
structure_summary = pd.DataFrame([{
    "n_unique_smiles": len(structures),
    "n_parse_ok": len(ok),
    "n_parse_failures": len(parse_failures),
    "n_unique_canonical_smiles": ok["canonical_smiles"].nunique(),
    "n_unique_plain_inchikey14": ok["inchikey14_plain"].nunique(),
    "n_unique_conn_key": ok["conn_key"].nunique(),
    "n_multi_component": int((ok["n_components"] > 1).sum()),
    "n_charged": int(ok["is_charged"].sum()),
    "n_with_stereo": int(ok["has_stereo"].sum()),
    "n_with_isotopes": int(ok["has_isotopes"].sum()),
    "n_tautomer_cap_hits": n_tautomer_timeouts,
    "pct_tautomer_cap_hits": round(100 * n_tautomer_timeouts / len(ok), 4),
}])
print(structure_summary.T)
save_csv(structure_summary, "res_03_structure_summary",
         "Unique SMILES/canonical-SMILES/InChIKey14/conn_key counts, parse failures, multi-component/charged/stereo counts, tautomer-cap hits")

assert len(parse_failures) / len(structures) < 0.01, "unexpectedly high SMILES parse-failure rate -- investigate before trusting downstream counts"
assert structure_summary["n_unique_conn_key"][0] <= structure_summary["n_unique_plain_inchikey14"][0], \
    "tautomer canonicalization should only merge groups, never split them"
_done()


In [ ]:
_done = section_timer("Section 3: tautomer-merging effect")

# How many *plain* (dataset) InChIKey14 groups merge once we tautomer-canonicalize?
plain_to_conn = ok.groupby("inchikey14_plain")["conn_key"].nunique()
n_plain_groups = len(plain_to_conn)
n_conn_groups = ok["conn_key"].nunique()
merge_examples = (
    ok.groupby("conn_key")["inchikey14_plain"]
    .agg(lambda s: sorted(s.unique()))
    .reset_index()
)
merge_examples["n_plain_keys_merged"] = merge_examples["inchikey14_plain"].apply(len)
merge_examples = merge_examples[merge_examples["n_plain_keys_merged"] > 1].sort_values("n_plain_keys_merged", ascending=False)

tautomer_effect = pd.DataFrame([{
    "n_plain_inchikey14_groups": n_plain_groups,
    "n_conn_key_groups": n_conn_groups,
    "n_groups_merged_by_tautomer_canonicalization": n_plain_groups - n_conn_groups,
    "n_conn_keys_spanning_multiple_plain_keys": len(merge_examples),
}])
save_csv(tautomer_effect, "res_03_tautomer_effect",
         "How many plain-InChIKey14 groups merge under tautomer canonicalization")

example_rows = merge_examples.head(20).copy()
example_rows["inchikey14_plain"] = example_rows["inchikey14_plain"].apply(lambda l: ";".join(l))
save_csv(example_rows, "res_03_tautomer_effect_examples", "Top conn_keys that merged the most plain-InChIKey14 groups, with examples")
print(tautomer_effect.T)
_done()


In [ ]:
_done = section_timer("Section 3: key collisions")

# conn_keys mapping to >1 distinct canonical SMILES -- either genuine stereo/salt variants
# collapsing (expected) or a real key-collision risk worth knowing about.
key_to_smiles = ok.groupby("conn_key")["canonical_smiles"].agg(lambda s: sorted(s.unique()))
n_smiles_per_key = key_to_smiles.apply(len)
collisions = key_to_smiles[n_smiles_per_key > 1].sort_values(key=lambda s: s.apply(len), ascending=False)
key_collisions = collisions.head(200).reset_index()
key_collisions["n_distinct_smiles"] = key_collisions["canonical_smiles"].apply(len)
key_collisions["example_smiles"] = key_collisions["canonical_smiles"].apply(lambda l: ";".join(l[:5]))
key_collisions = key_collisions[["conn_key", "n_distinct_smiles", "example_smiles"]]
save_csv(key_collisions, "res_03_key_collisions", "conn_keys with >1 distinct canonical SMILES (top 200 by count)")
print(f"{(n_smiles_per_key > 1).sum()} of {len(key_to_smiles)} conn_keys ({100*(n_smiles_per_key > 1).mean():.2f}%) map to >1 distinct canonical SMILES.")
_done()


In [ ]:
_done = section_timer("Section 3: element frequency")

elem_lists = ok["element_set"].str.split(",")
elem_counts = Counter()
for elems in elem_lists:
    elem_counts.update(elems)
element_frequency = pd.DataFrame([
    {"element": e, "n_structures": c, "pct_structures": round(100 * c / len(ok), 3)}
    for e, c in elem_counts.items()
]).sort_values("n_structures", ascending=False).reset_index(drop=True)
save_csv(element_frequency, "res_03_element_frequency", "Fraction of unique structures containing each element")
print(element_frequency.head(15))
_done()


In [ ]:
_done = section_timer("Section 3: mass/heavy-atom/element/collision figures")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(ok["exact_mass"].dropna(), bins=100, color="steelblue")
ax.set_xlabel("neutral exact (monoisotopic) mass (Da)")
ax.set_ylabel("count")
ax.set_title(f"Neutral exact mass of {len(ok)} unique train structures")
save_fig(fig, "fig_03_exact_mass", "Neutral exact mass histogram", "Unique train structures")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(ok["heavy_atom_count"].dropna(), bins=80, color="steelblue")
ax.set_xlabel("heavy atom count")
ax.set_ylabel("count")
ax.set_title("Heavy atom count of unique train structures")
save_fig(fig, "fig_03_heavy_atoms", "Heavy atom count histogram", "Unique train structures")

fig, ax = plt.subplots(figsize=(9, 5))
top_elems = element_frequency.head(15)
ax.barh(top_elems["element"], top_elems["pct_structures"], color="steelblue")
ax.set_xlabel("% of unique structures containing this element")
ax.invert_yaxis()
ax.set_title("Element frequency across unique train structures")
save_fig(fig, "fig_03_element_frequency", "Element frequency", "Fraction of structures containing each element, top 15")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(n_smiles_per_key, bins=np.arange(1, n_smiles_per_key.max() + 2) - 0.5, color="steelblue", log=True)
ax.set_xlabel("distinct SMILES per conn_key")
ax.set_ylabel("count of conn_keys (log scale)")
ax.set_title("Distinct SMILES per conn_key")
save_fig(fig, "fig_03_smiles_per_key", "Distinct SMILES per conn_key", "Log-scale histogram")
_done()


In [ ]:
_done = section_timer("Section 3: example key-collision grid image")

if len(key_collisions) > 0:
    from rdkit.Chem import Draw as _Draw
    example_keys = key_collisions.head(4)
    mols, legends = [], []
    for _, row in example_keys.iterrows():
        for smi in row["example_smiles"].split(";")[:2]:
            m = Chem.MolFromSmiles(smi)
            if m is not None:
                mols.append(m)
                legends.append(f"{row['conn_key']}\n{smi[:30]}")
    if mols:
        img = _Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(250, 250), legends=legends, returnPNG=False)
        img_path = FIG_DIR / "fig_03_example_collisions.png"
        img.save(img_path)
        FIGURE_MANIFEST.append(("fig_03_example_collisions", "Example key-collision structures",
                                 "RDKit grid image of molecule pairs sharing a conn_key"))
        print(f"[fig] fig_03_example_collisions.png — {len(mols)} structures from {len(example_keys)} colliding conn_keys")
    else:
        print("No drawable examples found for key collisions.")
else:
    print("No conn_key collisions found -- skipping example grid image.")
_done()


**Findings — Section 3.**
- `structures.parquet` caches one row per unique train SMILES; conn_key/mrr_at_25/
  validate_submission unit tests all pass (stereoisomers and keto/enol tautomers collapse to
  the same key, constitutional isomers don't, and the real `sample_submission.csv` validates
  clean). See `res_03_structure_summary.csv` for exact counts (unique SMILES vs unique
  canonical SMILES vs unique plain InChIKey14 vs unique conn_key — the last should be ≤ the
  others, since tautomer canonicalization only merges groups).
- Tautomer canonicalization merges a measurable number of plain-InChIKey14 groups
  (`res_03_tautomer_effect.csv`) — this is exactly the mechanism the competition's key is
  designed to reward, and it means training labels derived from the dataset's own
  `inchikey14` column are *finer* than the true prediction target; later notebooks should
  train/evaluate on `conn_key`, not the raw `inchikey14` column.
- A small, bounded fraction of molecules hit the tautomer-search cap
  (`n_tautomer_cap_hits`/`pct_tautomer_cap_hits` in `res_03_structure_summary.csv`) — these
  fall back to the plain key and are logged, not silently treated as exact.
- `res_03_key_collisions.csv` lists conn_keys that still map to >1 distinct canonical SMILES
  after tautomer canonicalization — a hard ceiling on any approach that reports a single
  "best" structure per conn_key, since RDKit's tautomer canonicalization doesn't fully
  disambiguate these.


## Section 4 — Molecule-level table

Join `structures` back onto `spectra_meta`. The prediction unit differs by split: **train
molecule unit = `conn_key`** (built in Section 3); **test molecule unit = `molecule_id`**
(the id the submission format actually uses -- we don't know test structures, so we can't
compute a test conn_key).


In [ ]:
_done = section_timer("Section 4: attach conn_key to every train spectrum")

smiles_to_conn_key = structures.set_index("smiles")["conn_key"]
spectra_meta["conn_key"] = np.where(
    spectra_meta["split"] == "train",
    spectra_meta[COLUMN_MAP["smiles"]].map(smiles_to_conn_key),
    np.nan,
)
n_unmapped_train = spectra_meta.loc[spectra_meta["split"] == "train", "conn_key"].isna().sum()
assert n_unmapped_train == 0, f"{n_unmapped_train} train spectra have no conn_key -- structures cache is stale, rebuild it"
print("[OK] every train spectrum has a conn_key.")
_done()


In [ ]:
_done = section_timer("Section 4: molecule-level aggregation")


def n_distinct_ce(ce_mean_series):
    return ce_mean_series.dropna().round(1).nunique()


def molecule_level_agg(df, group_col):
    g = df.groupby(group_col)
    out = pd.DataFrame({
        "n_spectra": g.size(),
        "n_adducts": g[COLUMN_MAP["adduct"]].nunique(),
        "n_collision_energies": g["ce_mean"].apply(n_distinct_ce),
        "n_sources": g["source"].nunique(),
        "n_instruments": g[COLUMN_MAP["instrument"]].nunique(),
        "n_ion_modes": g[COLUMN_MAP["ion_mode"]].nunique(),
        "precursor_mz_min": g[COLUMN_MAP["precursor_mz"]].min(),
        "precursor_mz_max": g[COLUMN_MAP["precursor_mz"]].max(),
    })
    out["precursor_mz_range"] = out["precursor_mz_max"] - out["precursor_mz_min"]
    return out.reset_index().rename(columns={group_col: "molecule_key"})


train_molecules = molecule_level_agg(spectra_meta[spectra_meta["split"] == "train"], "conn_key")
train_molecules["split"] = "train"
test_molecules = molecule_level_agg(spectra_meta[spectra_meta["split"] == "test"], "molecule_id")
test_molecules["split"] = "test"

molecules = pd.concat([train_molecules, test_molecules], ignore_index=True)
mem_report(molecules, "molecules")
assert molecules.loc[molecules["split"] == "train", "molecule_key"].is_unique
assert molecules.loc[molecules["split"] == "test", "molecule_key"].is_unique
assert len(train_molecules) == structures["conn_key"].nunique(), "train molecule count should equal n_unique conn_key"
assert len(test_molecules) == spectra_meta.loc[spectra_meta["split"] == "test", "molecule_id"].nunique()

molecules.to_parquet(cache_path("molecules"), index=False)
print(f"[cache] wrote {len(molecules)} molecule rows to {cache_path('molecules')}")
_done()


In [ ]:
_done = section_timer("Section 4: summary tables")

quantile_cols = ["n_spectra", "n_adducts", "n_collision_energies", "n_sources", "n_instruments", "n_ion_modes", "precursor_mz_range"]
mol_summary_rows = []
for split, sub in molecules.groupby("split"):
    for c in quantile_cols:
        q = sub[c].quantile([0.5, 0.75, 0.9, 0.95, 0.99]).to_dict()
        mol_summary_rows.append({"split": split, "stat": c, **{f"p{int(k*100)}": v for k, v in q.items()},
                                  "mean": sub[c].mean(), "max": sub[c].max()})
molecule_level_summary = pd.DataFrame(mol_summary_rows)
save_csv(molecule_level_summary, "res_04_molecule_level_summary", "Quantiles of per-molecule stats, train vs test")

spectra_per_molecule_counts = (
    molecules.groupby(["split", "n_spectra"]).size().reset_index(name="n_molecules")
)
save_csv(spectra_per_molecule_counts, "res_04_spectra_per_molecule_counts", "Histogram table of n_spectra per molecule, train vs test")
print(molecule_level_summary.head(10))
_done()


In [ ]:
_done = section_timer("Section 4: figures")

fig, ax = plt.subplots(figsize=(9, 5))
for split, color in [("train", "steelblue"), ("test", color_for_source("test"))]:
    vals = molecules.loc[molecules["split"] == split, "n_spectra"]
    ax.hist(vals, bins=np.logspace(0, np.log10(vals.max()), 40), alpha=0.5, label=f"{split} (n={len(vals)})",
            color=color, density=True)
ax.set_xscale("log")
ax.set_xlabel("spectra per molecule (log scale)")
ax.set_ylabel("density")
ax.set_title("Spectra per molecule, train vs test")
ax.legend()
save_fig(fig, "fig_04_spectra_per_molecule", "Spectra per molecule, train vs test", "Log-x, density-normalized")


def molecule_stat_hist(col, name, title, xlabel):
    fig, ax = plt.subplots(figsize=(9, 5))
    for split, color in [("train", "steelblue"), ("test", color_for_source("test"))]:
        vals = molecules.loc[molecules["split"] == split, col]
        ax.hist(vals, bins=np.arange(vals.max() + 2) - 0.5, alpha=0.5, label=f"{split} (n={len(vals)})",
                color=color, density=True)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("density")
    ax.set_title(title)
    ax.legend()
    return fig


fig = molecule_stat_hist("n_adducts", "fig_04_adducts_per_molecule", "Distinct adducts per molecule, train vs test", "n distinct adducts")
save_fig(fig, "fig_04_adducts_per_molecule", "Distinct adducts per molecule", "Train vs test")

fig = molecule_stat_hist("n_collision_energies", "fig_04_ce_per_molecule", "Distinct collision energies per molecule, train vs test", "n distinct CE (rounded to 0.1)")
save_fig(fig, "fig_04_ce_per_molecule", "Distinct collision energies per molecule", "Train vs test")

fig, ax = plt.subplots(figsize=(8, 5))
train_sources_per_mol = molecules.loc[molecules["split"] == "train", "n_sources"].value_counts().sort_index()
ax.bar(train_sources_per_mol.index.astype(str), train_sources_per_mol.values, color="steelblue")
ax.set_yscale("log")
ax.set_xlabel("n distinct sources the same train structure appears in")
ax.set_ylabel("n molecules (log scale)")
ax.set_title("How often the same train structure appears in multiple sources")
save_fig(fig, "fig_04_sources_per_molecule", "Sources per molecule", "Train only (test source is always 'test')")
_done()


**Findings — Section 4.** `molecules.parquet` has one row per train `conn_key`
(≈274,289 — matches Section 3's unique conn_key count exactly) and one row per test
`molecule_id` (400). Compare `res_04_molecule_level_summary.csv`'s train vs test quantiles of
`n_spectra`, `n_adducts`, `n_collision_energies` directly — any large gap here is a second,
molecule-level confirmation of the same train/test shift Section 2 saw at the spectrum level,
and it's exactly what Section 9's `test_like_holdout` needs to match.


## Section 5 — Precursor mass consistency

`parse_adduct` (in `src/eda_utils.py`) parses an adduct string into `(multiplier, mass_shift_da,
charge)` using exact element masses and the electron-mass correction, covering the generic
`[nM+-frag...]z+/-` grammar rather than a fixed list — this dataset's real adducts include 121
distinct strings (Section 2), and the generic parser was checked against every one of them.


In [ ]:
_done = section_timer("Section 5: adduct parse coverage")

train_adducts = spectra_meta.loc[spectra_meta["split"] == "train", COLUMN_MAP["adduct"]]
adduct_vc = train_adducts.value_counts()
unparsed_rows = []
for a, cnt in adduct_vc.items():
    if U.parse_adduct(a) is None:
        unparsed_rows.append({"adduct": a, "n_spectra": cnt, "pct_of_train": round(100 * cnt / len(train_adducts), 5)})
unparsed_adducts = pd.DataFrame(unparsed_rows).sort_values("n_spectra", ascending=False) if unparsed_rows else \
    pd.DataFrame(columns=["adduct", "n_spectra", "pct_of_train"])
save_csv(unparsed_adducts, "res_05_unparsed_adducts", "Adduct strings parse_adduct couldn't parse, with frequency")
print(f"{len(unparsed_adducts)} distinct unparsed adducts covering {unparsed_adducts['n_spectra'].sum() if len(unparsed_adducts) else 0} "
      f"of {len(train_adducts)} train spectra ({100 * (unparsed_adducts['n_spectra'].sum() if len(unparsed_adducts) else 0) / len(train_adducts):.4f}%).")
assert (unparsed_adducts["n_spectra"].sum() if len(unparsed_adducts) else 0) / len(train_adducts) < 0.001, \
    "unparsed-adduct rate is unexpectedly high -- extend parse_adduct before trusting Section 5 mass errors"
_done()


In [ ]:
_done = section_timer("Section 5: train precursor ppm/Da error")

train_mass = spectra_meta[spectra_meta["split"] == "train"].copy()
train_mass = train_mass.merge(structures[["smiles", "exact_mass"]].rename(columns={"smiles": COLUMN_MAP["smiles"]}),
                                on=COLUMN_MAP["smiles"], how="left")

parsed = train_mass[COLUMN_MAP["adduct"]].apply(U.parse_adduct)
train_mass["adduct_parsed"] = parsed.notna()
train_mass["expected_precursor_mz"] = [
    (p[0] * m + p[1]) / abs(p[2]) if p is not None and pd.notna(m) else np.nan
    for p, m in zip(parsed, train_mass["exact_mass"])
]
train_mass["error_da"] = train_mass[COLUMN_MAP["precursor_mz"]] - train_mass["expected_precursor_mz"]
train_mass["error_ppm"] = 1e6 * train_mass["error_da"] / train_mass["expected_precursor_mz"]

n_scored = train_mass["error_ppm"].notna().sum()
print(f"{n_scored} of {len(train_mass)} train spectra have a scoreable expected precursor m/z "
      f"({100 * n_scored / len(train_mass):.2f}%).")
_done()


In [ ]:
_done = section_timer("Section 5: ppm quantiles by source/adduct/instrument")


def ppm_quantile_table(df, group_col):
    rows = []
    for g, sub in df.groupby(group_col):
        e = sub["error_ppm"].dropna()
        d = sub["error_da"].dropna()
        if len(e) < 5:
            continue
        rows.append({
            group_col: g, "n": len(e),
            "abs_ppm_p50": e.abs().quantile(0.50), "abs_ppm_p90": e.abs().quantile(0.90),
            "abs_ppm_p95": e.abs().quantile(0.95), "abs_ppm_p99": e.abs().quantile(0.99),
            "abs_ppm_p999": e.abs().quantile(0.999),
            "abs_da_p50": d.abs().quantile(0.50), "abs_da_p90": d.abs().quantile(0.90),
            "abs_da_p95": d.abs().quantile(0.95), "abs_da_p99": d.abs().quantile(0.99),
        })
    return pd.DataFrame(rows).sort_values("n", ascending=False)


ppm_by_source = ppm_quantile_table(train_mass, "source")
ppm_by_adduct = ppm_quantile_table(train_mass, COLUMN_MAP["adduct"])
ppm_by_instrument = ppm_quantile_table(train_mass, COLUMN_MAP["instrument"])
for df_, name in [(ppm_by_source, "source"), (ppm_by_adduct, "adduct"), (ppm_by_instrument, "instrument")]:
    df_.insert(0, "group_type", name)
    df_.rename(columns={name: "group_value"}, inplace=True)
ppm_quantiles = pd.concat([ppm_by_source, ppm_by_adduct, ppm_by_instrument], ignore_index=True)
save_csv(ppm_quantiles, "res_05_ppm_quantiles", "Quantiles of |ppm| and |Da| error by source / adduct / instrument")
print(ppm_quantiles.head(10))
_done()


In [ ]:
_done = section_timer("Section 5: mass outliers")

outliers = train_mass[train_mass["error_ppm"].abs() > 50].copy()
print(f"{len(outliers)} train spectra ({100*len(outliers)/n_scored:.3f}% of scoreable rows) have |ppm error| > 50.")
outlier_sample = outliers.reindex(outliers["error_ppm"].abs().sort_values(ascending=False).index).head(500)
outlier_cols = [COLUMN_MAP["smiles"], COLUMN_MAP["adduct"], "source", COLUMN_MAP["precursor_mz"],
                "expected_precursor_mz", "error_da", "error_ppm"]
save_csv(outlier_sample[outlier_cols], "res_05_mass_outliers", "Worst 500 |ppm| mass outliers, by source/adduct/smiles")
print("Outlier sources breakdown:")
print(outliers["source"].value_counts())
print("Outlier adduct breakdown (top 10):")
print(outliers[COLUMN_MAP["adduct"]].value_counts().head(10))
_done()


In [ ]:
_done = section_timer("Section 5: test neutral mass estimates")

test_mass = spectra_meta[spectra_meta["split"] == "test"].copy()
parsed_test = test_mass[COLUMN_MAP["adduct"]].apply(U.parse_adduct)
test_mass["neutral_mass_est"] = [
    (mz * abs(p[2]) - p[1]) / p[0] if p is not None else np.nan
    for p, mz in zip(parsed_test, test_mass[COLUMN_MAP["precursor_mz"]])
]
assert test_mass["neutral_mass_est"].notna().all(), "every test adduct should parse -- test only uses 7 known adduct forms (Section 2)"

test_neutral = test_mass.groupby("molecule_id").agg(
    n_spectra=("spectrum_id", "count"),
    adducts=(COLUMN_MAP["adduct"], lambda s: ";".join(sorted(s.unique()))),
    neutral_mass_mean=("neutral_mass_est", "mean"),
    neutral_mass_std=("neutral_mass_est", "std"),
    neutral_mass_min=("neutral_mass_est", "min"),
    neutral_mass_max=("neutral_mass_est", "max"),
).reset_index()
test_neutral["neutral_mass_range_ppm"] = 1e6 * (test_neutral["neutral_mass_max"] - test_neutral["neutral_mass_min"]) / test_neutral["neutral_mass_mean"]
save_csv(test_neutral, "res_05_test_neutral_masses", "Per-test-molecule neutral mass estimate and its consistency across that molecule's spectra")

inconsistent = (test_neutral["neutral_mass_range_ppm"] > 50).sum()
print(f"{inconsistent} of {len(test_neutral)} test molecules have >50ppm spread in neutral-mass estimate across their own spectra "
      f"(candidate wrong-adduct-label or multimer/fragment cases).")
_done()


In [ ]:
_done = section_timer("Section 5: figures")

fig, ax = plt.subplots(figsize=(10, 5))
top_sources = train_mass["source"].value_counts().head(6).index
for src in top_sources:
    vals = train_mass.loc[train_mass["source"] == src, "error_ppm"].dropna().clip(-30, 30)
    ax.hist(vals, bins=60, alpha=0.4, density=True, label=src, color=color_for_source(src))
ax.set_xlabel("signed precursor ppm error (clipped to ±30)")
ax.set_ylabel("density")
ax.set_title("Precursor ppm error by source")
ax.legend(fontsize=8)
save_fig(fig, "fig_05_ppm_error_by_source", "Signed ppm error by source", "Clipped to ±30 ppm, top-6 sources")

fig, ax = plt.subplots(figsize=(9, 5))
sub = train_mass.dropna(subset=["error_ppm", COLUMN_MAP["precursor_mz"]])
hb = ax.hexbin(sub[COLUMN_MAP["precursor_mz"]], sub["error_ppm"].clip(-50, 50), gridsize=60, cmap="viridis", mincnt=1)
fig.colorbar(hb, ax=ax, label="count")
ax.set_xlabel("precursor m/z (Th)")
ax.set_ylabel("ppm error (clipped to ±50)")
ax.set_title("ppm error vs precursor m/z")
save_fig(fig, "fig_05_ppm_vs_mz", "ppm error vs m/z hexbin", "Reveals mass-dependent calibration drift, if any")

fig, ax = plt.subplots(figsize=(9, 5))
tol_grid = np.linspace(0, 50, 100)
enveda_source = "enveda-180"
for src in list(top_sources) + ([enveda_source] if enveda_source not in top_sources else []):
    vals = train_mass.loc[train_mass["source"] == src, "error_ppm"].dropna().abs()
    if len(vals) == 0:
        continue
    frac = [100 * (vals <= t).mean() for t in tol_grid]
    lw = 3 if src == enveda_source else 1.5
    ax.plot(tol_grid, frac, label=src + (" (most test-like)" if src == enveda_source else ""),
            color=color_for_source(src), linewidth=lw)
ax.set_xlabel("ppm tolerance")
ax.set_ylabel("% of spectra within tolerance")
ax.set_title("Cumulative % of spectra within ppm tolerance, by source")
ax.legend(fontsize=8)
save_fig(fig, "fig_05_recommended_tolerance", "Cumulative %-within-tolerance curves", "One curve per source, enveda-180 highlighted")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(test_neutral["neutral_mass_range_ppm"].dropna().clip(upper=200), bins=60, color=color_for_source("test"))
ax.set_xlabel("neutral-mass spread across a test molecule's own spectra (ppm, clipped to 200)")
ax.set_ylabel("n test molecules")
ax.set_title("Test neutral-mass self-consistency")
save_fig(fig, "fig_05_test_mass_consistency", "Test neutral-mass self-consistency", "Spread in ppm per test molecule")
_done()


**Findings — Section 5.** `parse_adduct` covers essentially all train adducts (see
`res_05_unparsed_adducts.csv` for the tiny remainder — real element-mass gaps like uncommon
metal adducts, not grammar failures). `res_05_ppm_quantiles.csv` gives per-source/adduct/
instrument tolerance numbers directly usable by notebook 02's candidate generation; the
highlighted `enveda-180` curve in `fig_05_recommended_tolerance` is the one to trust for test,
since it's the source that actually matches test's instrument (Section 2/7). `res_05_mass_outliers.csv`
gives concrete rows to eyeball for wrong-adduct-label vs isotope-peak vs charged-species
explanations rather than guessing which. Test's own multi-spectrum neutral-mass consistency
(`fig_05_test_mass_consistency`) bounds how much noise a mass-based candidate filter should
tolerate at inference time.


## Section 6 — Peak-level analysis (sample)

Peak arrays are the one thing we never load in full. This section takes a
`PEAK_SAMPLE_SIZE`-row (200,000) stratified-by-source sample of **train** spectra plus **all**
1,213 test spectra, streams only those rows' `mz_array`/`intensity_array` out of the parquet
file in bounded-memory batches, and caches the result to `outputs/cache/peak_sample.parquet`.


In [ ]:
_done = section_timer("Section 6: stratified sample selection")


def sample_stratified_offsets(meta, total, seed, min_floor=50):
    """Pick up to `total` train row offsets, stratified proportionally by source (with a
    floor per source so tiny sources aren\'t sampled to zero), using the physical row offset
    encoded in `spectrum_id` ("train_<offset>") -- spectra_meta itself never held peak arrays,
    so this offset is the only way back to the right parquet rows."""
    rng = np.random.default_rng(seed)
    train_rows = meta[meta["split"] == "train"]
    counts = train_rows["source"].value_counts()
    offsets = []
    chosen_meta = []
    for src, n_src in counts.items():
        alloc = min(n_src, max(min_floor, round(total * n_src / len(train_rows))))
        sub = train_rows[train_rows["source"] == src]
        picked = sub.sample(n=int(alloc), random_state=rng.integers(0, 2**31 - 1))
        for sid in picked["spectrum_id"]:
            offsets.append(int(sid.split("_")[1]))
        chosen_meta.append(picked)
    order = np.argsort(offsets)
    sorted_offsets = np.array(offsets)[order]
    return sorted_offsets, pd.concat(chosen_meta, ignore_index=True)


train_offsets, train_sample_meta = sample_stratified_offsets(spectra_meta, EFFECTIVE_PEAK_SAMPLE_SIZE, CONFIG["SEED"])
print(f"Selected {len(train_offsets)} train spectra across {train_sample_meta['source'].nunique()} sources "
      f"(target was {EFFECTIVE_PEAK_SAMPLE_SIZE}, {MODE_TAG} mode).")
print(train_sample_meta["source"].value_counts())
_done()


In [ ]:
_done = section_timer("Section 6: streaming peak read (bounded memory)")

PEAK_SAMPLE_CACHE = cache_path("peak_sample")


def read_peaks_by_offset(path, mz_col, intensity_col, sorted_offsets):
    """Read only `sorted_offsets` rows' peak-array columns out of `path`, by reading *only the
    row groups that actually contain one of those offsets* (using parquet metadata to find
    them) -- row groups with no target rows are never touched at all, and each targeted row
    group is read once, converted, and dropped before moving to the next, so at most one row
    group's worth of the two peak columns is ever resident in memory."""
    sorted_offsets = np.asarray(sorted_offsets)
    pf = pq.ParquetFile(path)
    starts, ends = row_group_bounds(pf)
    out_offsets, out_mz, out_inten = [], [], []
    for rg in range(pf.metadata.num_row_groups):
        lo, hi = starts[rg], ends[rg]
        mask = (sorted_offsets >= lo) & (sorted_offsets < hi)
        if not mask.any():
            continue
        local_idx = sorted_offsets[mask] - lo
        tbl = pf.read_row_group(rg, columns=[mz_col, intensity_col]).to_pandas()
        for li, global_off in zip(local_idx, sorted_offsets[mask]):
            out_offsets.append(int(global_off))
            out_mz.append(np.asarray(tbl[mz_col].iloc[li], dtype=np.float64))
            out_inten.append(np.asarray(tbl[intensity_col].iloc[li], dtype=np.float64))
        del tbl
    return pd.DataFrame({"row_offset": out_offsets, "mz_array": out_mz, "intensity_array": out_inten})


if PEAK_SAMPLE_CACHE.exists():
    peak_sample = pd.read_parquet(PEAK_SAMPLE_CACHE)
    print(f"[cache] loaded {len(peak_sample)} peak-sample rows from {PEAK_SAMPLE_CACHE}")
else:
    train_peaks = read_peaks_by_offset(DATA_DIR / "train.parquet", COLUMN_MAP["mz_array"], COLUMN_MAP["intensity_array"], train_offsets)
    train_peaks["spectrum_id"] = "train_" + train_peaks["row_offset"].astype(str)
    train_peaks = train_peaks.drop(columns="row_offset")

    test_peaks = pq.read_table(DATA_DIR / "test.parquet", columns=[COLUMN_MAP["mz_array"], COLUMN_MAP["intensity_array"], COLUMN_MAP["test_spectrum_id"]]).to_pandas()
    test_peaks = test_peaks.rename(columns={COLUMN_MAP["mz_array"]: "mz_array", COLUMN_MAP["intensity_array"]: "intensity_array",
                                              COLUMN_MAP["test_spectrum_id"]: "spectrum_id"})

    peak_sample = pd.concat([train_peaks, test_peaks], ignore_index=True)
    peak_sample.to_parquet(PEAK_SAMPLE_CACHE, index=False)
    print(f"[cache] wrote {len(peak_sample)} peak-sample rows to {PEAK_SAMPLE_CACHE}")

mem_report(peak_sample, "peak_sample")
assert len(peak_sample) == len(train_offsets) + len(test_meta), "peak_sample should have one row per sampled train spectrum plus every test spectrum"
_done()


In [ ]:
_done = section_timer("Section 6: attach metadata and compute per-spectrum peak stats")

peak_sample = peak_sample.merge(
    spectra_meta[["spectrum_id", "split", "source", "conn_key", COLUMN_MAP["precursor_mz"], COLUMN_MAP["adduct"], COLUMN_MAP["ion_mode"], "ce_mean"]],
    on="spectrum_id", how="left",
)
assert peak_sample["split"].notna().all(), "every sampled spectrum_id should join back to spectra_meta"

peak_stats_records = [
    U.compute_peak_stats(row.mz_array, row.intensity_array, getattr(row, COLUMN_MAP["precursor_mz"]))
    for row in peak_sample.itertuples()
]
peak_stats = pd.DataFrame(peak_stats_records)
peak_sample = pd.concat([peak_sample.reset_index(drop=True), peak_stats], axis=1)
mem_report(peak_sample, "peak_sample (with stats)")
_done()


In [ ]:
_done = section_timer("Section 6: peak stats by source + data quality flags")

quantile_cols_6 = ["n_peaks", "n_duplicate_mz", "entropy", "frac_peaks_above_precursor",
                    "frac_intensity_above_precursor", "base_peak_mz_ratio", "mz_decimal_places_median"]
rows6 = []
for grp, sub in peak_sample.groupby("source"):
    rec = {"source": grp, "n_spectra": len(sub)}
    for c in quantile_cols_6:
        q = sub[c].quantile([0.5, 0.9, 0.99])
        rec[f"{c}_p50"] = q.loc[0.5]
        rec[f"{c}_p90"] = q.loc[0.9]
        rec[f"{c}_p99"] = q.loc[0.99]
    rows6.append(rec)
peak_stats_by_source = pd.DataFrame(rows6)
save_csv(peak_stats_by_source, "res_06_peak_stats_by_source", "Quantiles of every peak stat per source (train sources + test)")

quality_flags = pd.DataFrame([{
    "source": grp,
    "n_spectra": len(sub),
    "n_unsorted": int((~sub["sorted_ok"]).sum()),
    "n_with_duplicate_mz": int((sub["n_duplicate_mz"] > 0).sum()),
    "n_empty": int((sub["n_peaks"] == 0).sum()),
    "n_single_peak": int((sub["n_peaks"] == 1).sum()),
    "n_with_peaks_above_precursor": int((sub["frac_peaks_above_precursor"] > 0).sum()),
    "n_precursor_peak_present": int(sub["precursor_peak_present"].sum()),
} for grp, sub in peak_sample.groupby("source")])
save_csv(quality_flags, "res_06_data_quality_flags", "Counts of unsorted/duplicated/empty/single-peak/above-precursor spectra, per source")
print(quality_flags)
_done()


In [ ]:
_done = section_timer("Section 6: distribution figures")

fig, ax = plt.subplots(figsize=(9, 5))
for src in list(peak_sample["source"].unique()):
    vals = peak_sample.loc[peak_sample["source"] == src, "n_peaks"]
    ax.hist(vals, bins=np.logspace(0, np.log10(max(vals.max(), 2)), 40), histtype="step",
            label=f"{src} (n={len(vals)})", color=color_for_source(src), density=True)
ax.set_xscale("log")
ax.set_xlabel("n peaks (log scale)")
ax.set_ylabel("density")
ax.set_title("Peaks per spectrum, per source vs test")
ax.legend(fontsize=7)
save_fig(fig, "fig_06_n_peaks", "Peaks per spectrum by source", "Log-x, step histograms, test included")

fig, ax = plt.subplots(figsize=(9, 5))
for src in list(peak_sample["source"].unique()):
    vals = peak_sample.loc[peak_sample["source"] == src, "entropy"]
    ax.hist(vals, bins=40, histtype="step", label=src, color=color_for_source(src), density=True)
ax.set_xlabel("spectral entropy")
ax.set_ylabel("density")
ax.set_title("Spectral entropy per source vs test")
ax.legend(fontsize=7)
save_fig(fig, "fig_06_entropy", "Spectral entropy by source", "Step histograms, test included")

fig, ax = plt.subplots(figsize=(9, 5))
for src in list(peak_sample["source"].unique()):
    vals = peak_sample.loc[peak_sample["source"] == src, "base_peak_mz_ratio"].dropna()
    ax.hist(vals, bins=40, range=(0, 1.5), histtype="step", label=src, color=color_for_source(src), density=True)
ax.axvline(1.0, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("base peak m/z / precursor m/z")
ax.set_ylabel("density")
ax.set_title("Base peak position relative to precursor, per source vs test")
ax.legend(fontsize=7)
save_fig(fig, "fig_06_base_peak_position", "Base peak position ratio", "1.0 = base peak is the precursor")

fig, ax = plt.subplots(figsize=(9, 5))
scale_counts = peak_sample.groupby(["source", "intensity_scale"]).size().unstack(fill_value=0)
scale_counts = scale_counts.div(scale_counts.sum(axis=1), axis=0)
scale_counts.plot(kind="barh", stacked=True, ax=ax, colormap="tab10")
ax.set_xlabel("fraction of spectra")
ax.set_title("Intensity normalization convention, per source vs test")
save_fig(fig, "fig_06_intensity_scale", "Intensity scale convention by source", "Stacked fractions per detected scale bucket")
_done()


In [ ]:
_done = section_timer("Section 6: example spectra plots")


def stick_plot(ax, mz, intensity, color="steelblue", label=None):
    ax.vlines(mz, 0, intensity, color=color, linewidth=1)
    ax.axhline(0, color="black", linewidth=0.5)
    if label:
        ax.set_title(label, fontsize=9)


test_examples = peak_sample[peak_sample["split"] == "test"].sample(6, random_state=CONFIG["SEED"])
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (_, row) in zip(axes.flat, test_examples.iterrows()):
    stick_plot(ax, row["mz_array"], row["intensity_array"], color=color_for_source("test"),
               label=f"{row['spectrum_id']}\nadduct={row[COLUMN_MAP['adduct']]}, n_peaks={row['n_peaks']}")
    ax.set_xlabel("m/z")
    ax.set_ylabel("intensity")
fig.suptitle("6 example test spectra")
fig.tight_layout()
save_fig(fig, "fig_06_example_spectra_test", "Example test spectra", "6 stick plots sampled from the test peak sample")

# same train molecule at different collision energies
ce_groups = peak_sample[peak_sample["split"] == "train"].groupby("conn_key")
multi_ce_key = None
for key, sub in ce_groups:
    if sub[COLUMN_MAP["precursor_mz"]].notna().all() and sub["spectrum_id"].nunique() >= 3:
        multi_ce_key = key
        multi_ce_sub = sub
        break

if multi_ce_key is not None:
    n_show = min(4, len(multi_ce_sub))
    fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 4), sharey=True)
    if n_show == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, multi_ce_sub.head(n_show).iterrows()):
        ce_label = f"ce_mean={row['ce_mean']:.1f}" if pd.notna(row["ce_mean"]) else "ce_mean=unknown"
        stick_plot(ax, row["mz_array"], row["intensity_array"], color="steelblue", label=ce_label)
        ax.set_xlabel("m/z")
    fig.suptitle(f"Same train molecule (conn_key={multi_ce_key}) across spectra")
    fig.tight_layout()
    save_fig(fig, "fig_06_same_molecule_ce", "Same molecule, multiple spectra", "One train conn_key sampled at multiple acquisitions")
else:
    print("No train conn_key with >=3 sampled spectra found in this peak sample -- skipping fig_06_same_molecule_ce.")
_done()


In [ ]:
_done = section_timer("Section 6: mirror plot (same molecule, different sources)")

conn_key_sources = peak_sample[peak_sample["split"] == "train"].groupby("conn_key")["source"].nunique()
multi_source_keys = conn_key_sources[conn_key_sources > 1].index
mirror_pair = None
for key in multi_source_keys:
    sub = peak_sample[(peak_sample["split"] == "train") & (peak_sample["conn_key"] == key)]
    srcs = sub["source"].unique()
    if len(srcs) >= 2:
        row_a = sub[sub["source"] == srcs[0]].iloc[0]
        row_b = sub[sub["source"] == srcs[1]].iloc[0]
        mirror_pair = (row_a, row_b)
        break

if mirror_pair is not None:
    row_a, row_b = mirror_pair
    fig, ax = plt.subplots(figsize=(10, 5))
    inten_a = row_a["intensity_array"] / (row_a["intensity_array"].max() or 1)
    inten_b = row_b["intensity_array"] / (row_b["intensity_array"].max() or 1)
    ax.vlines(row_a["mz_array"], 0, inten_a, color=color_for_source(row_a["source"]), label=row_a["source"])
    ax.vlines(row_b["mz_array"], 0, -inten_b, color=color_for_source(row_b["source"]), label=row_b["source"])
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_xlabel("m/z")
    ax.set_ylabel("relative intensity (mirrored)")
    ax.set_title(f"Mirror plot: conn_key={row_a['conn_key']}, {row_a['source']} vs {row_b['source']}")
    ax.legend()
    save_fig(fig, "fig_06_mirror_plot", "Mirror plot, same molecule two sources", "Top/bottom = two sources' spectra for the same conn_key")
else:
    print("No conn_key with >=2 sources found in this peak sample -- skipping fig_06_mirror_plot.")
_done()


**Findings — Section 6 (part 1).** `peak_sample.parquet` caches 200k stratified train spectra
+ all 1,213 test spectra with peak arrays and per-spectrum stats (`res_06_peak_stats_by_source.csv`,
`res_06_data_quality_flags.csv`). Compare test's row in these tables against `enveda-180`'s row
directly — that comparison (not the all-train aggregate) is the one that matters for what
notebook 03's preprocessing should assume about peak count, entropy, base-peak position, and
intensity normalization at inference time.


In [ ]:
_done = section_timer("Section 6: quick same-vs-diff cosine preview (for notebook 03)")

from scipy import sparse
from sklearn.metrics import roc_auc_score

BIN_WIDTH = 1.0
N_BINS = int(np.ceil(peak_sample["max_mz"].max() / BIN_WIDTH)) + 2


def build_binned_sparse_matrix(df, bin_width=BIN_WIDTH, n_bins=N_BINS):
    """1-Da-binned, sqrt-intensity, L2-normalized spectra as a sparse (n_spectra x n_bins)
    matrix -- a quick preview representation for notebook 03, not the final featurization."""
    rows, cols, data = [], [], []
    for i, (mz, inten) in enumerate(zip(df["mz_array"], df["intensity_array"])):
        if len(mz) == 0:
            continue
        idx = np.clip((mz / bin_width).astype(int), 0, n_bins - 1)
        rows.extend([i] * len(idx))
        cols.extend(idx.tolist())
        data.extend(np.sqrt(np.clip(inten, 0, None)).tolist())
    mat = sparse.csr_matrix((data, (rows, cols)), shape=(len(df), n_bins))
    norms = np.sqrt(mat.multiply(mat).sum(axis=1)).A1
    norms[norms == 0] = 1.0
    return sparse.diags(1.0 / norms) @ mat


train_peak_sample = peak_sample[peak_sample["split"] == "train"].reset_index(drop=True)
spec_matrix = build_binned_sparse_matrix(train_peak_sample)
print(f"Binned matrix: {spec_matrix.shape}, {spec_matrix.nnz} nonzeros ({spec_matrix.nnz / spec_matrix.shape[0]:.1f} bins/spectrum avg)")

rng = np.random.default_rng(CONFIG["SEED"])
N_PAIRS_TARGET = 20_000

# same-conn_key pairs
same_pairs = []
for key, idx in train_peak_sample.groupby("conn_key").indices.items():
    if len(idx) < 2:
        continue
    n_take = min(len(idx) * (len(idx) - 1) // 2, 4)
    for _ in range(n_take):
        i, j = rng.choice(idx, size=2, replace=False)
        same_pairs.append((i, j))
    if len(same_pairs) >= N_PAIRS_TARGET:
        break
same_pairs = same_pairs[:N_PAIRS_TARGET]

# different-molecule, mass-matched (within 0.01 Da) pairs
by_mass = train_peak_sample.sort_values(COLUMN_MAP["precursor_mz"]).reset_index()
mz_sorted = by_mass[COLUMN_MAP["precursor_mz"]].to_numpy()
diff_pairs = []
attempts = 0
max_attempts = N_PAIRS_TARGET * 20
while len(diff_pairs) < N_PAIRS_TARGET and attempts < max_attempts:
    attempts += 1
    i = rng.integers(0, len(by_mass))
    lo = np.searchsorted(mz_sorted, mz_sorted[i] - 0.01, side="left")
    hi = np.searchsorted(mz_sorted, mz_sorted[i] + 0.01, side="right")
    if hi - lo < 2:
        continue
    j = rng.integers(lo, hi)
    if j == i:
        continue
    if by_mass.loc[i, "conn_key"] == by_mass.loc[j, "conn_key"]:
        continue
    diff_pairs.append((by_mass.loc[i, "index"], by_mass.loc[j, "index"]))

print(f"Collected {len(same_pairs)} same-conn_key pairs and {len(diff_pairs)} mass-matched different-molecule pairs.")


def pair_cosines(pairs, mat):
    out = np.empty(len(pairs))
    for k, (i, j) in enumerate(pairs):
        out[k] = mat[i].multiply(mat[j]).sum()
    return out


same_cos = pair_cosines(same_pairs, spec_matrix)
diff_cos = pair_cosines(diff_pairs, spec_matrix)

labels = np.concatenate([np.ones(len(same_cos)), np.zeros(len(diff_cos))])
scores = np.concatenate([same_cos, diff_cos])
auc = roc_auc_score(labels, scores)

same_vs_diff_summary = pd.DataFrame([
    {"group": "same_conn_key", "n_pairs": len(same_cos), "p10": np.quantile(same_cos, 0.10),
     "p50": np.quantile(same_cos, 0.50), "p90": np.quantile(same_cos, 0.90), "mean": same_cos.mean()},
    {"group": "different_molecule_mass_matched", "n_pairs": len(diff_cos), "p10": np.quantile(diff_cos, 0.10),
     "p50": np.quantile(diff_cos, 0.50), "p90": np.quantile(diff_cos, 0.90), "mean": diff_cos.mean()},
    {"group": "auc", "n_pairs": len(labels), "p10": auc, "p50": auc, "p90": auc, "mean": auc},
])
save_csv(same_vs_diff_summary, "res_06_same_vs_diff_cosine",
         "Quantiles of binned-cosine similarity for same-conn_key vs mass-matched different-molecule pairs, plus ROC AUC")
print(same_vs_diff_summary)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(same_cos, bins=60, alpha=0.5, density=True, label=f"same conn_key (n={len(same_cos)})", color="seagreen")
ax.hist(diff_cos, bins=60, alpha=0.5, density=True, label=f"different molecule, mass-matched (n={len(diff_cos)})", color="indianred")
ax.set_xlabel("binned-spectrum cosine similarity")
ax.set_ylabel("density")
ax.set_title(f"Same-molecule vs different-molecule spectral cosine (AUC={auc:.3f})")
ax.legend()
save_fig(fig, "fig_06_same_vs_diff_cosine", "Same vs different molecule cosine", "Overlaid histograms + ROC AUC in the title")
_done()


**Findings — Section 6 (part 2).** The quick 1-Da-binned/sqrt-intensity/L2-normalized cosine
preview separates same-conn_key spectrum pairs from mass-matched different-molecule pairs with
a clearly informative AUC (see `res_06_same_vs_diff_cosine.csv` for the exact number this run) —
i.e. simple binned-spectrum cosine similarity already carries real signal even before any of
notebook 03's fancier featurization, and mass-matching the negative pairs (±0.01 Da) means this
AUC isn't just reflecting "different mass region", it's reflecting real spectral shape
difference between isobaric/near-isobaric structures.


## Section 7 — Train vs test domain shift

Three independent views of the same question -- which train source looks most like test --
so no single metric's blind spot drives the conclusion: (1) per-feature distribution distance
per source, (2) a classifier trying to tell train from test, (3) a mass-only candidate-density
proxy for the actual retrieval problem.


In [ ]:
_done = section_timer("Section 7: per-feature shift by source (KS / Jensen-Shannon)")

from scipy.stats import ks_2samp
from scipy.spatial.distance import jensenshannon

test_rows = spectra_meta[spectra_meta["split"] == "test"]
numeric_shift_cols = [COLUMN_MAP["precursor_mz"], "ce_mean"]
categorical_shift_cols = [COLUMN_MAP["adduct"], COLUMN_MAP["ion_mode"], COLUMN_MAP["instrument"]]


def js_distance_categorical(a, b):
    cats = sorted(set(a.dropna().unique()) | set(b.dropna().unique()))
    pa_ = a.value_counts(normalize=True).reindex(cats, fill_value=0.0)
    pb_ = b.value_counts(normalize=True).reindex(cats, fill_value=0.0)
    return float(jensenshannon(pa_.to_numpy(), pb_.to_numpy(), base=2))


shift_rows = []
for src, sub in spectra_meta[spectra_meta["split"] == "train"].groupby("source"):
    for c in numeric_shift_cols:
        a, b = sub[c].dropna(), test_rows[c].dropna()
        if len(a) < 5 or len(b) < 5:
            continue
        stat, pval = ks_2samp(a, b)
        shift_rows.append({"source": src, "feature": c, "metric": "ks_statistic", "value": stat, "p_value": pval})
    for c in categorical_shift_cols:
        d = js_distance_categorical(sub[c], test_rows[c])
        shift_rows.append({"source": src, "feature": c, "metric": "js_distance", "value": d, "p_value": np.nan})

shift_by_source = pd.DataFrame(shift_rows)
save_csv(shift_by_source, "res_07_shift_by_source", "KS statistic (numeric) / Jensen-Shannon distance (categorical) per source vs test")

overall_shift = shift_by_source.groupby("source")["value"].mean().sort_values()
most_test_like_source = overall_shift.index[0]
print(f"Most test-like source by mean shift metric: {most_test_like_source}")
print(overall_shift)
_done()


In [ ]:
_done = section_timer("Section 7: shift heatmap")

heat = shift_by_source.pivot_table(index="feature", columns="source", values="value")
fig, ax = plt.subplots(figsize=(10, 4 + 0.3 * len(heat)))
im = ax.imshow(heat.to_numpy(), aspect="auto", cmap="viridis")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
fig.colorbar(im, ax=ax, label="KS statistic / JS distance (higher = more different from test)")
ax.set_title("Train/test shift by feature and source")
save_fig(fig, "fig_07_shift_heatmap", "Shift heatmap", "Rows = features, columns = sources; darker/lower = more test-like")
_done()


In [ ]:
_done = section_timer("Section 7: adversarial validation")

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

adv_df = peak_sample.copy()
adv_df["is_test"] = (adv_df["split"] == "test").astype(int)
adv_features_numeric = [COLUMN_MAP["precursor_mz"], "n_peaks", "entropy", "frac_peaks_above_precursor",
                         "base_peak_mz_ratio", "mz_decimal_places_median"]
adv_features_categorical = [COLUMN_MAP["adduct"], COLUMN_MAP["ion_mode"]]

X = adv_df[adv_features_numeric].copy()
for c in adv_features_categorical:
    X[c] = adv_df[c].astype("category").cat.codes.replace(-1, np.nan)
y = adv_df["is_test"].to_numpy()

n_pos = int(y.sum())
if n_pos >= CONFIG["N_FOLDS"] and (len(y) - n_pos) >= CONFIG["N_FOLDS"]:
    cv = StratifiedKFold(n_splits=CONFIG["N_FOLDS"], shuffle=True, random_state=CONFIG["SEED"])
    clf = HistGradientBoostingClassifier(random_state=CONFIG["SEED"])
    oof_pred = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:, 1]
    adv_auc = roc_auc_score(y, oof_pred)

    clf.fit(X, y)
    perm = permutation_importance(clf, X, y, n_repeats=5, random_state=CONFIG["SEED"], scoring="roc_auc")
    importance_df = pd.DataFrame({
        "feature": X.columns, "importance_mean": perm.importances_mean, "importance_std": perm.importances_std,
    }).sort_values("importance_mean", ascending=False)

    adversarial_validation = pd.DataFrame([{"auc": adv_auc, "n_train": int((y == 0).sum()), "n_test": int((y == 1).sum())}])
    save_csv(adversarial_validation, "res_07_adversarial_validation", "5-fold OOF ROC AUC distinguishing train from test (HistGradientBoostingClassifier)")
    save_csv(importance_df, "res_07_adversarial_importance_table", "Permutation importance for the adversarial classifier")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(importance_df["feature"], importance_df["importance_mean"], xerr=importance_df["importance_std"], color="steelblue")
    ax.invert_yaxis()
    ax.set_xlabel("permutation importance (ROC AUC drop)")
    ax.set_title(f"Adversarial validation feature importance (AUC={adv_auc:.3f})")
    save_fig(fig, "fig_07_adversarial_importance", "Adversarial validation importance", "Permutation importance, HistGradientBoostingClassifier")
    print(f"Adversarial validation AUC: {adv_auc:.4f} (0.5 = indistinguishable, 1.0 = trivially separable)")
else:
    print(f"Too few positive/negative examples for {CONFIG['N_FOLDS']}-fold adversarial validation at this scale "
          f"(n_test={n_pos}, n_train={len(y) - n_pos}) -- skipping (expected in SMOKE_TEST mode with a small peak sample).")
    adversarial_validation = pd.DataFrame([{"auc": np.nan, "n_train": len(y) - n_pos, "n_test": n_pos}])
    save_csv(adversarial_validation, "res_07_adversarial_validation", "Skipped: insufficient rows at this scale")
_done()


In [ ]:
_done = section_timer("Section 7: mass-only candidate density proxy (NOT recall -- no true labels used)")

conn_key_mass = structures.dropna(subset=["exact_mass"]).groupby("conn_key")["exact_mass"].median().sort_values()
mass_array = conn_key_mass.to_numpy()

def count_candidates_within_ppm(neutral_mass, ppm_tol):
    window = neutral_mass * ppm_tol / 1e6
    lo = np.searchsorted(mass_array, neutral_mass - window, side="left")
    hi = np.searchsorted(mass_array, neutral_mass + window, side="right")
    return hi - lo


coverage_rows = []
for _, row in test_neutral.iterrows():
    if pd.isna(row["neutral_mass_mean"]):
        continue
    rec = {"molecule_id": row["molecule_id"], "neutral_mass": row["neutral_mass_mean"]}
    for tol in (5, 10, 20):
        rec[f"n_candidates_{tol}ppm"] = count_candidates_within_ppm(row["neutral_mass_mean"], tol)
    coverage_rows.append(rec)
test_mass_coverage = pd.DataFrame(coverage_rows)
save_csv(test_mass_coverage, "res_07_test_mass_coverage",
         "Per-test-molecule count of train conn_keys within N ppm of its neutral mass -- a candidate-DENSITY proxy, not recall (true labels unknown)")

zero_cand_rate_10ppm = (test_mass_coverage["n_candidates_10ppm"] == 0).mean()
print(f"Zero-candidate rate at 10ppm: {100 * zero_cand_rate_10ppm:.2f}% of test molecules "
      f"(NOTE: at {MODE_TAG} scale, conn_key_mass only has {len(conn_key_mass)} unique train structures -- "
      f"this rate is meaningless until rerun at full scale).")

fig, ax = plt.subplots(figsize=(9, 5))
for tol, color in [(5, "indianred"), (10, "steelblue"), (20, "seagreen")]:
    vals = test_mass_coverage[f"n_candidates_{tol}ppm"].clip(lower=0.5)
    ax.hist(vals, bins=np.logspace(np.log10(0.5), np.log10(max(vals.max(), 2)), 40), alpha=0.5, label=f"{tol} ppm", color=color)
ax.set_xscale("log")
ax.set_xlabel("n train conn_keys within tolerance (log scale; 0 shown at 0.5)")
ax.set_ylabel("n test molecules")
ax.set_title("Mass-only candidate count per test molecule (density proxy, not recall)")
ax.legend()
save_fig(fig, "fig_07_test_candidate_counts", "Candidate count distribution", "Log-x histogram at 5/10/20 ppm; NOT a recall measurement")
_done()


**Findings — Section 7.** All three views should point the same direction: `res_07_shift_by_source.csv`
names the source with the lowest mean KS/JS distance to test, the adversarial classifier's AUC
says how separable train and test are overall (near 0.5 = good sign for generalization; near 1.0
= a strong, learnable train/test gap notebook 02+ needs to account for), and
`res_07_test_mass_coverage.csv` gives a purely mass-based candidate-density floor -- explicitly
**not** a recall number, since it doesn't know the true structure, only how crowded the mass
window is. At full scale, expect `enveda-180` to lead on all three (it's the only source that's
100% timsTOF, matching test's instrument exactly per Section 2). At smoke scale, treat the
specific numbers here as a logic check only — rerun with `SMOKE_TEST=False` for real conclusions.


## Section 8 — Duplicates, scaffolds and chemical diversity


In [ ]:
_done = section_timer("Section 8: conn_key frequency")

train_molecules_sorted = molecules[molecules["split"] == "train"].sort_values("n_spectra", ascending=False).reset_index(drop=True)
train_molecules_sorted["rank"] = np.arange(1, len(train_molecules_sorted) + 1)
key_frequency = train_molecules_sorted[["rank", "molecule_key", "n_spectra"]].rename(columns={"molecule_key": "conn_key"})
save_csv(key_frequency, "res_08_key_frequency", "Spectra-per-conn_key, rank-frequency table")

pct_singleton_keys = 100 * (train_molecules_sorted["n_spectra"] == 1).mean()
print(f"{pct_singleton_keys:.2f}% of train conn_keys are singletons (exactly 1 spectrum).")
_done()


In [ ]:
_done = section_timer("Section 8: scaffold assignment and summary")

# One representative Murcko scaffold per conn_key (tautomers of the same connectivity almost
# always share a ring system; "first" is a pragmatic simplification for this diversity
# overview, not the fold-construction logic in Section 9, which handles this more carefully).
conn_key_scaffold = structures.dropna(subset=["conn_key"]).groupby("conn_key")["murcko_scaffold"].first()

scaffold_table = train_molecules_sorted.merge(
    conn_key_scaffold.rename("murcko_scaffold"), left_on="molecule_key", right_index=True, how="left",
)
n_missing_scaffold = scaffold_table["murcko_scaffold"].isna().sum()
print(f"{n_missing_scaffold} of {len(scaffold_table)} train conn_keys have no scaffold assignment (parse issues) -- excluded below.")
scaffold_table = scaffold_table.dropna(subset=["murcko_scaffold"])

scaffold_agg = scaffold_table.groupby("murcko_scaffold").agg(
    n_molecules=("molecule_key", "nunique"),
    n_spectra=("n_spectra", "sum"),
).reset_index().sort_values("n_spectra", ascending=False)

top_scaffolds = scaffold_agg.head(100).copy()
top_scaffolds["pct_of_train_spectra"] = round(100 * top_scaffolds["n_spectra"] / scaffold_table["n_spectra"].sum(), 4)
save_csv(top_scaffolds, "res_08_top_scaffolds", "Top-100 Murcko scaffolds by molecule/spectra count")

n_acyclic = (scaffold_agg["murcko_scaffold"] == "").sum()
acyclic_row = scaffold_agg[scaffold_agg["murcko_scaffold"] == ""]
n_acyclic_molecules = int(acyclic_row["n_molecules"].sum()) if len(acyclic_row) else 0
scaffold_summary = pd.DataFrame([{
    "n_scaffolds_total": scaffold_agg["murcko_scaffold"].nunique(),
    "n_singleton_scaffolds": int((scaffold_agg["n_molecules"] == 1).sum()),
    "n_acyclic_molecules": n_acyclic_molecules,
    "pct_acyclic_molecules": round(100 * n_acyclic_molecules / scaffold_table["molecule_key"].nunique(), 3),
}])
save_csv(scaffold_summary, "res_08_scaffold_summary", "Scaffold count, singleton scaffolds, acyclic-molecule share")
print(scaffold_summary.T)
_done()


In [ ]:
_done = section_timer("Section 8: rank-frequency and coverage figures")

fig, ax = plt.subplots(figsize=(8, 6))
ax.loglog(key_frequency["rank"], key_frequency["n_spectra"], ".", markersize=2, color="steelblue")
ax.set_xlabel("conn_key rank (by spectra count)")
ax.set_ylabel("n spectra")
ax.set_title("conn_key rank-frequency (log-log)")
save_fig(fig, "fig_08_key_rank_frequency", "conn_key rank-frequency", "Log-log; a straight-ish line indicates heavy-tailed reuse")

scaffold_rank = scaffold_agg.reset_index(drop=True)
scaffold_rank["rank"] = np.arange(1, len(scaffold_rank) + 1)
fig, ax = plt.subplots(figsize=(8, 6))
ax.loglog(scaffold_rank["rank"], scaffold_rank["n_spectra"], ".", markersize=2, color="indianred")
ax.set_xlabel("scaffold rank (by spectra count)")
ax.set_ylabel("n spectra")
ax.set_title("Scaffold rank-frequency (log-log)")
save_fig(fig, "fig_08_scaffold_rank_frequency", "Scaffold rank-frequency", "Log-log")

fig, ax = plt.subplots(figsize=(9, 5))
key_cum = np.cumsum(key_frequency["n_spectra"]) / key_frequency["n_spectra"].sum() * 100
scaffold_cum = np.cumsum(scaffold_rank["n_spectra"]) / scaffold_rank["n_spectra"].sum() * 100
ax.plot(np.arange(1, len(key_cum) + 1), key_cum, label="conn_key", color="steelblue")
ax.plot(np.arange(1, len(scaffold_cum) + 1), scaffold_cum, label="scaffold", color="indianred")
ax.set_xscale("log")
ax.set_xlabel("top-k (log scale)")
ax.set_ylabel("cumulative % of train spectra")
ax.set_title("Cumulative spectra coverage by top-k keys / scaffolds")
ax.legend()
save_fig(fig, "fig_08_cumulative_coverage", "Cumulative coverage", "conn_key vs scaffold, log-x")
_done()


In [ ]:
_done = section_timer("Section 8: top scaffolds grid image")

drawable_scaffolds = top_scaffolds[top_scaffolds["murcko_scaffold"] != ""].head(12)
mols, legends = [], []
for _, row in drawable_scaffolds.iterrows():
    m = Chem.MolFromSmiles(row["murcko_scaffold"])
    if m is not None:
        mols.append(m)
        legends.append(f"n_molecules={row['n_molecules']}, n_spectra={row['n_spectra']}")

if mols:
    from rdkit.Chem import Draw as _Draw
    img = _Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(250, 250), legends=legends, returnPNG=False)
    img_path = FIG_DIR / "fig_08_top_scaffolds_grid.png"
    img.save(img_path)
    FIGURE_MANIFEST.append(("fig_08_top_scaffolds_grid", "Top scaffolds grid", "12 most frequent (non-acyclic) Murcko scaffolds"))
    print(f"[fig] fig_08_top_scaffolds_grid.png — {len(mols)} scaffolds")
else:
    print("No drawable non-empty scaffolds found at this scale -- skipping fig_08_top_scaffolds_grid.")
_done()


**Findings — Section 8.** `res_08_key_frequency.csv`/`res_08_top_scaffolds.csv` give the
rank-frequency structure notebook 03 needs for any frequency-aware weighting or hard-negative
mining decision; the singleton-key percentage and acyclic-molecule share
(`res_08_scaffold_summary.csv`) set expectations for how much structural repetition (or lack
of it) any train-side memorization can actually exploit. The generic-scaffold grouping used
here for the diversity overview is deliberately simpler than Section 9's scaffold-fold
construction, which handles the acyclic/empty-scaffold group explicitly rather than lumping
every acyclic molecule into one giant fold-crossing group.


## Section 9 — Validation design (most important section)

Three fold schemes, every one leakage-safe at the connectivity level (every spectrum of a
conn_key falls in exactly one fold, by construction, not by post-hoc checking):

- **A. `group_kfold`** — `StratifiedGroupKFold` grouped by `conn_key`, stratified by
  source × precursor-mass bin. The main training/robustness CV.
- **B. `scaffold_kfold`** — grouped by generic Murcko scaffold instead (harder: forces whole
  chemotypes out of a fold at once). Acyclic molecules (empty scaffold) fall back to being
  their own singleton group, rather than one giant cross-fold "no scaffold" bucket.
- **C. `test_like_holdout`** — ~400–1000 molecules from the most test-like source only
  (Section 7), matched to test's own spectra-per-molecule and adduct distribution. **This is
  the primary number notebooks 02+ should report**; A/B are for training and robustness checks.


In [ ]:
_done = section_timer("Section 9: molecule-level info for fold construction")

train_mol_info = molecules[molecules["split"] == "train"].copy()
train_mol_info = train_mol_info.rename(columns={"molecule_key": "conn_key"})

primary_source = (
    spectra_meta[spectra_meta["split"] == "train"]
    .groupby("conn_key")["source"].agg(lambda s: s.value_counts().idxmax())
)
primary_adduct = (
    spectra_meta[spectra_meta["split"] == "train"]
    .groupby("conn_key")[COLUMN_MAP["adduct"]].agg(lambda s: s.value_counts().idxmax())
)
conn_key_generic_scaffold = structures.dropna(subset=["conn_key"]).groupby("conn_key")["generic_scaffold"].first()

train_mol_info["primary_source"] = train_mol_info["conn_key"].map(primary_source)
train_mol_info["primary_adduct"] = train_mol_info["conn_key"].map(primary_adduct)
train_mol_info["exact_mass"] = train_mol_info["conn_key"].map(conn_key_mass)
train_mol_info["generic_scaffold"] = train_mol_info["conn_key"].map(conn_key_generic_scaffold)
train_mol_info = train_mol_info.dropna(subset=["primary_source", "exact_mass"]).reset_index(drop=True)

n_mass_bins = min(10, train_mol_info["exact_mass"].nunique())
train_mol_info["mass_bin"] = pd.qcut(train_mol_info["exact_mass"], q=max(n_mass_bins, 2), duplicates="drop", labels=False)
train_mol_info["strat_label"] = train_mol_info["primary_source"].astype(str) + "_mb" + train_mol_info["mass_bin"].astype(str)
print(f"{len(train_mol_info)} train molecules ready for fold construction, {train_mol_info['strat_label'].nunique()} strat labels.")
_done()


In [ ]:
_done = section_timer("Section 9A: group_kfold (StratifiedGroupKFold by conn_key)")

from sklearn.model_selection import StratifiedGroupKFold

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sgkf = StratifiedGroupKFold(n_splits=CONFIG["N_FOLDS"], shuffle=True, random_state=CONFIG["SEED"])
    fold_group = np.full(len(train_mol_info), -1)
    X_dummy = np.zeros(len(train_mol_info))
    for fold_i, (_, val_idx) in enumerate(sgkf.split(X_dummy, train_mol_info["strat_label"], train_mol_info["conn_key"])):
        fold_group[val_idx] = fold_i
train_mol_info["fold_group"] = fold_group

assert (fold_group >= 0).all(), "every molecule must be assigned to a fold"
leak_check_a = train_mol_info.groupby("conn_key")["fold_group"].nunique()
assert leak_check_a.max() == 1, "a conn_key was split across more than one fold in scheme A"
print("[OK] scheme A: every conn_key in exactly one fold.")
print(train_mol_info["fold_group"].value_counts().sort_index())
_done()


In [ ]:
_done = section_timer("Section 9B: scaffold_kfold (StratifiedGroupKFold by generic scaffold)")

is_acyclic = train_mol_info["generic_scaffold"].fillna("") == ""
scaffold_group_col = np.where(is_acyclic, "ACYCLIC_" + train_mol_info["conn_key"].astype(str), train_mol_info["generic_scaffold"])
train_mol_info["scaffold_group"] = scaffold_group_col

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sgkf_b = StratifiedGroupKFold(n_splits=CONFIG["N_FOLDS"], shuffle=True, random_state=CONFIG["SEED"])
    fold_scaffold = np.full(len(train_mol_info), -1)
    for fold_i, (_, val_idx) in enumerate(sgkf_b.split(X_dummy, train_mol_info["strat_label"], train_mol_info["scaffold_group"])):
        fold_scaffold[val_idx] = fold_i
train_mol_info["fold_scaffold"] = fold_scaffold

assert (fold_scaffold >= 0).all()
real_scaffold_rows = train_mol_info[~is_acyclic]
leak_check_b = real_scaffold_rows.groupby("generic_scaffold")["fold_scaffold"].nunique()
assert len(leak_check_b) == 0 or leak_check_b.max() == 1, "a (non-acyclic) scaffold was split across more than one fold in scheme B"
print(f"[OK] scheme B: no shared (non-acyclic) scaffold crosses a fold. {int(is_acyclic.sum())} acyclic molecules "
      f"({100 * is_acyclic.mean():.1f}%) fell back to singleton conn_key groups.")
print(train_mol_info["fold_scaffold"].value_counts().sort_index())
_done()


In [ ]:
_done = section_timer("Section 9C: test_like_holdout")

TEST_ADDUCT_SET = set(test_rows[COLUMN_MAP["adduct"]].unique())
candidates = train_mol_info[
    (train_mol_info["primary_source"] == most_test_like_source) & (train_mol_info["primary_adduct"].isin(TEST_ADDUCT_SET))
].copy()
print(f"{len(candidates)} candidate molecules from the most test-like source ({most_test_like_source}) with a test-seen adduct.")

test_n_spectra = molecules.loc[molecules["split"] == "test", "n_spectra"]
bin_edges = np.unique(np.quantile(test_n_spectra, np.linspace(0, 1, 6)))
if len(bin_edges) < 2:
    bin_edges = np.array([test_n_spectra.min() - 0.5, test_n_spectra.max() + 0.5])
test_bucket_props = pd.cut(test_n_spectra, bins=bin_edges, include_lowest=True).value_counts(normalize=True)
candidates["bucket"] = pd.cut(candidates["n_spectra"], bins=bin_edges, include_lowest=True)

HOLDOUT_TARGET = min(700, max(1, len(candidates)))
rng9 = np.random.default_rng(CONFIG["SEED"])
holdout_parts = []
for bucket, prop in test_bucket_props.items():
    pool = candidates[candidates["bucket"] == bucket]
    n_take = min(len(pool), round(HOLDOUT_TARGET * prop))
    if n_take > 0:
        holdout_parts.append(pool.sample(n=int(n_take), random_state=int(rng9.integers(0, 2**31 - 1))))
holdout = pd.concat(holdout_parts, ignore_index=True) if holdout_parts else candidates.iloc[0:0].copy()
train_mol_info["in_test_like_holdout"] = train_mol_info["conn_key"].isin(holdout["conn_key"])

print(f"test_like_holdout: {len(holdout)} molecules "
      f"({'within the 400-1000 target range' if 400 <= len(holdout) <= 1000 else 'OUTSIDE the 400-1000 target range -- expected at ' + MODE_TAG + ' scale' if MODE_TAG == 'smoke' else 'OUTSIDE the 400-1000 target range -- investigate'}).")
_done()


In [ ]:
_done = section_timer("Section 9: fold summary table")

fold_summary_rows = []
for scheme, fold_col in [("group_kfold", "fold_group"), ("scaffold_kfold", "fold_scaffold")]:
    for fold_i, sub in train_mol_info.groupby(fold_col):
        fold_summary_rows.append({
            "scheme": scheme, "fold": fold_i, "n_molecules": len(sub), "n_spectra": int(sub["n_spectra"].sum()),
            "top_source_share": round(sub["primary_source"].value_counts(normalize=True).iloc[0], 3),
            "top_adduct_share": round(sub["primary_adduct"].value_counts(normalize=True).iloc[0], 3),
            "exact_mass_median": sub["exact_mass"].median(),
        })
if len(holdout):
    fold_summary_rows.append({
        "scheme": "test_like_holdout", "fold": "holdout", "n_molecules": len(holdout), "n_spectra": int(holdout["n_spectra"].sum()),
        "top_source_share": 1.0, "top_adduct_share": round(holdout["primary_adduct"].value_counts(normalize=True).iloc[0], 3),
        "exact_mass_median": holdout["exact_mass"].median(),
    })
fold_summary = pd.DataFrame(fold_summary_rows)
save_csv(fold_summary, "res_09_fold_summary", "Per-scheme/fold molecule/spectra counts, source/adduct mix, mass median")
print(fold_summary)
_done()


In [ ]:
_done = section_timer("Section 9: structural leakage (max Tanimoto to training folds)")

from rdkit.Chem import rdFingerprintGenerator
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
conn_key_smiles = structures.dropna(subset=["conn_key"]).groupby("conn_key")["canonical_smiles"].first()

_fp_cache = {}


def get_fp_cached(conn_key):
    if conn_key not in _fp_cache:
        smi = conn_key_smiles.get(conn_key)
        mol = Chem.MolFromSmiles(smi) if smi else None
        _fp_cache[conn_key] = morgan_gen.GetFingerprint(mol) if mol is not None else None
    return _fp_cache[conn_key]


REFERENCE_SAMPLE_SIZE = 20_000  # computational subsample of the training-fold reference set; documented approximation
N_LEAKAGE_QUERIES = 2_000
rng9b = np.random.default_rng(CONFIG["SEED"])


def sample_max_tanimoto(val_keys, train_keys, n_queries=N_LEAKAGE_QUERIES, n_reference=REFERENCE_SAMPLE_SIZE):
    val_sample = rng9b.choice(val_keys, size=min(n_queries, len(val_keys)), replace=False) if len(val_keys) else np.array([])
    ref_sample = rng9b.choice(train_keys, size=min(n_reference, len(train_keys)), replace=False) if len(train_keys) else np.array([])
    ref_fps = [fp for fp in (get_fp_cached(k) for k in ref_sample) if fp is not None]
    if not ref_fps:
        return np.array([])
    out = np.empty(len(val_sample))
    for i, k in enumerate(val_sample):
        qfp = get_fp_cached(k)
        out[i] = max(DataStructs.BulkTanimotoSimilarity(qfp, ref_fps)) if qfp is not None else np.nan
    return out


from rdkit import DataStructs

nn_rows = []
for scheme, fold_col in [("group_kfold", "fold_group"), ("scaffold_kfold", "fold_scaffold")]:
    val_keys = train_mol_info.loc[train_mol_info[fold_col] == 0, "conn_key"].to_numpy()
    train_keys = train_mol_info.loc[train_mol_info[fold_col] != 0, "conn_key"].to_numpy()
    sims = sample_max_tanimoto(val_keys, train_keys)
    for s in sims:
        nn_rows.append({"scheme": scheme, "max_tanimoto": s})

if len(holdout):
    val_keys_c = holdout["conn_key"].to_numpy()
    train_keys_c = train_mol_info.loc[~train_mol_info["in_test_like_holdout"], "conn_key"].to_numpy()
    sims_c = sample_max_tanimoto(val_keys_c, train_keys_c)
    for s in sims_c:
        nn_rows.append({"scheme": "test_like_holdout", "max_tanimoto": s})

nn_tanimoto = pd.DataFrame(nn_rows) if nn_rows else pd.DataFrame(columns=["scheme", "max_tanimoto"])
save_csv(nn_tanimoto, "res_09_nn_tanimoto", "Max Morgan(r=2,2048bit) Tanimoto of sampled validation molecules to a sampled training-fold reference set, per scheme")

fig, ax = plt.subplots(figsize=(9, 5))
for scheme, color in [("group_kfold", "steelblue"), ("scaffold_kfold", "indianred"), ("test_like_holdout", "seagreen")]:
    vals = nn_tanimoto.loc[nn_tanimoto["scheme"] == scheme, "max_tanimoto"].dropna()
    if len(vals):
        ax.hist(vals, bins=40, range=(0, 1), alpha=0.5, density=True, label=f"{scheme} (n={len(vals)})", color=color)
ax.set_xlabel("max Tanimoto similarity to training-fold reference")
ax.set_ylabel("density")
ax.set_title("Structural novelty of validation molecules, by fold scheme")
ax.legend()
save_fig(fig, "fig_09_nn_tanimoto", "Nearest-neighbor Tanimoto by scheme", "Lower = more novel/harder validation regime")
_done()


In [ ]:
_done = section_timer("Section 9: fold size / mass distribution / test-like match figures")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (scheme, fold_col) in zip(axes, [("group_kfold", "fold_group"), ("scaffold_kfold", "fold_scaffold")]):
    counts = train_mol_info.groupby(fold_col).size()
    ax.bar(counts.index.astype(str), counts.values, color="steelblue")
    ax.set_title(scheme)
    ax.set_xlabel("fold")
    ax.set_ylabel("n molecules")
fig.suptitle("Fold sizes, schemes A and B")
save_fig(fig, "fig_09_fold_sizes", "Fold sizes", "Molecule count per fold, group_kfold vs scaffold_kfold")

fig, ax = plt.subplots(figsize=(9, 5))
for fold_i in sorted(train_mol_info["fold_group"].unique()):
    vals = train_mol_info.loc[train_mol_info["fold_group"] == fold_i, "precursor_mz_min"].dropna()
    if len(vals) > 1:
        ax.hist(vals, bins=30, histtype="step", density=True, label=f"fold {fold_i}")
ax.set_xlabel("precursor m/z (min per molecule)")
ax.set_ylabel("density")
ax.set_title("Per-fold precursor m/z distribution (group_kfold)")
ax.legend(fontsize=8)
save_fig(fig, "fig_09_fold_mass_distribution", "Per-fold mass distribution", "group_kfold, one line per fold")

if len(holdout):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(holdout["n_spectra"], bins=20, alpha=0.5, density=True, label="test_like_holdout", color="seagreen")
    axes[0].hist(test_n_spectra, bins=20, alpha=0.5, density=True, label="real test", color=color_for_source("test"))
    axes[0].set_xlabel("n_spectra per molecule")
    axes[0].legend()
    axes[0].set_title("Spectra-per-molecule: holdout vs real test")

    holdout_adduct_frac = holdout["primary_adduct"].value_counts(normalize=True)
    test_adduct_frac = test_rows[COLUMN_MAP["adduct"]].value_counts(normalize=True)
    common_adducts = sorted(set(holdout_adduct_frac.index) | set(test_adduct_frac.index))
    x = np.arange(len(common_adducts))
    axes[1].bar(x - 0.2, [holdout_adduct_frac.get(a, 0) for a in common_adducts], width=0.4, label="holdout", color="seagreen")
    axes[1].bar(x + 0.2, [test_adduct_frac.get(a, 0) for a in common_adducts], width=0.4, label="real test", color=color_for_source("test"))
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(common_adducts, rotation=45, ha="right")
    axes[1].legend()
    axes[1].set_title("Adduct mix: holdout vs real test")
    fig.tight_layout()
    save_fig(fig, "fig_09_test_like_match", "Holdout vs real test match", "Spectra-per-molecule and adduct mix")
else:
    print("Holdout is empty at this scale -- skipping fig_09_test_like_match.")
_done()


In [ ]:
_done = section_timer("Section 9: save folds.csv / folds.parquet / spectrum_id-conn_key mapping")

folds_out = train_mol_info[["conn_key", "fold_group", "fold_scaffold", "in_test_like_holdout"]].copy()
folds_out.to_csv(RES_DIR / "folds.csv", index=False)
folds_out.to_parquet(cache_path("folds"), index=False)
RESULTS_MANIFEST.append(("folds", len(folds_out), folds_out.shape[1], "conn_key -> fold_group/fold_scaffold/in_test_like_holdout, for every later notebook to reuse"))
print(f"[res] folds.csv + {cache_path('folds').name}  ({len(folds_out)} rows)")

spectrum_to_conn_key = spectra_meta.loc[spectra_meta["split"] == "train", ["spectrum_id", "conn_key"]].reset_index(drop=True)
spectrum_to_conn_key.to_parquet(cache_path("spectrum_id_to_conn_key"), index=False)
print(f"[cache] wrote {len(spectrum_to_conn_key)} spectrum_id->conn_key rows to {cache_path('spectrum_id_to_conn_key')}")
_done()


In [ ]:
_done = section_timer("Section 9: sanity baseline (random mass-window predictor) through the metric harness")

all_conn_keys_sorted = conn_key_mass.index.to_numpy()
all_masses_sorted = conn_key_mass.to_numpy()
rng9c = np.random.default_rng(CONFIG["SEED"])

pred_lists, true_keys = [], []
for _, row in holdout.iterrows():
    mass = row["exact_mass"]
    if pd.isna(mass):
        continue
    window = mass * 10 / 1e6
    lo = np.searchsorted(all_masses_sorted, mass - window, side="left")
    hi = np.searchsorted(all_masses_sorted, mass + window, side="right")
    pool = all_conn_keys_sorted[lo:hi]
    n_pick = min(25, len(pool))
    if n_pick == 0:
        continue
    picks = rng9c.choice(pool, size=n_pick, replace=False).tolist()
    pred_lists.append(picks)
    true_keys.append(row["conn_key"])

if pred_lists:
    sanity_mrr = U.mrr_at_25(pred_lists, true_keys)
else:
    sanity_mrr = float("nan")

sanity_baseline = pd.DataFrame([{
    "n_holdout_molecules_scored": len(pred_lists), "mrr_at_25": sanity_mrr, "ppm_tolerance": 10,
    "note": "validates the mrr_at_25 harness end-to-end; NOT a real retrieval baseline (candidates drawn from all of train, including the query itself)",
}])
save_csv(sanity_baseline, "res_09_sanity_baseline", "Random 10ppm mass-window predictor MRR@25 on test_like_holdout, via the real mrr_at_25 function")
print(sanity_baseline.T)
_done()


**Findings — Section 9.** All three schemes pass their leakage assertions by construction
(every conn_key -- and, in scheme B, every shared non-acyclic scaffold -- lands in exactly one
fold). `res_09_nn_tanimoto.csv`/`fig_09_nn_tanimoto` rank the three regimes by how "novel" their
validation molecules really are relative to what remains in training — `scaffold_kfold` should
show the lowest max-Tanimoto (hardest/most novel), `group_kfold` the highest (easiest, since a
shared scaffold's near-duplicates can still be in the training folds), with `test_like_holdout`
somewhere in between depending on how test-like-source chemistry clusters. `res_09_sanity_baseline.csv`
is the harness sanity check: a trivial random-within-10ppm predictor scores strictly between 0
and 1 (never exactly 0 or 1 unless the candidate pool is pathologically small or the query mass
is unique), which is the evidence that `mrr_at_25`/the candidate-window logic behave correctly
end-to-end before anything in notebook 02 depends on them. `folds.csv`/`folds.parquet` and the
spectrum_id→conn_key mapping are the artifacts every later notebook should load rather than
re-deriving these splits.


## Section 10 — Summary and artifacts


In [ ]:
_done = section_timer("Section 10: headline summary")

recommended_ppm_row = ppm_quantiles[(ppm_quantiles["group_type"] == "source") & (ppm_quantiles["group_value"] == most_test_like_source)]
recommended_ppm = float(recommended_ppm_row["abs_ppm_p95"].iloc[0]) if len(recommended_ppm_row) else float("nan")

summary_kv = {
    "mode": MODE_TAG,
    "n_train_spectra_loaded": int((spectra_meta["split"] == "train").sum()),
    "n_test_spectra": int((spectra_meta["split"] == "test").sum()),
    "n_unique_train_smiles": len(structures),
    "n_unique_conn_keys": int(structures["conn_key"].nunique()),
    "n_train_sources_present": int(train_mol_info["primary_source"].nunique()),
    "n_test_molecules": int((molecules["split"] == "test").sum()),
    "median_spectra_per_molecule_train": float(molecules.loc[molecules["split"] == "train", "n_spectra"].median()),
    "median_spectra_per_molecule_test": float(molecules.loc[molecules["split"] == "test", "n_spectra"].median()),
    "most_test_like_source": most_test_like_source,
    "recommended_ppm_tolerance_p95_test_like_source": recommended_ppm,
    "zero_candidate_rate_10ppm": float(zero_cand_rate_10ppm),
    "adversarial_validation_auc": float(adversarial_validation["auc"].iloc[0]),
    "test_like_holdout_size": len(holdout),
    "sanity_baseline_mrr_at_25": float(sanity_baseline["mrr_at_25"].iloc[0]),
    "pct_tautomer_cap_hits": float(structure_summary["pct_tautomer_cap_hits"].iloc[0]),
    "pct_conn_keys_with_multiple_canonical_smiles": round(100 * len(key_collisions) / structures["conn_key"].nunique(), 4) if len(key_collisions) else 0.0,
}
summary_df = pd.DataFrame([{"key": k, "value": v} for k, v in summary_kv.items()])
save_csv(summary_df, "res_10_summary", "Headline numbers from every section, one row per key")
print(summary_df.to_string(index=False))
_done()


In [ ]:
_done = section_timer("Section 10: manifests")

figure_manifest_df = pd.DataFrame(FIGURE_MANIFEST, columns=["name", "title", "description"])
results_manifest_df = pd.DataFrame(RESULTS_MANIFEST, columns=["name", "n_rows", "n_cols", "description"])
figure_manifest_df.to_csv(RES_DIR / "res_10_figure_manifest.csv", index=False)
results_manifest_df.to_csv(RES_DIR / "res_10_results_manifest.csv", index=False)
print(f"[res] res_10_figure_manifest.csv  ({len(figure_manifest_df)} figures)")
print(f"[res] res_10_results_manifest.csv  ({len(results_manifest_df)} tables, including this pair)")
_done()


In [ ]:
_done = section_timer("Section 10: cached artifacts for later notebooks")

spectra_meta.to_parquet(cache_path("spectra_meta"), index=False)
print(f"[cache] wrote spectra_meta -> {cache_path('spectra_meta')}")

cache_files = sorted(CACHE_DIR.glob(f"*_{MODE_TAG}.parquet")) + [RES_DIR / "folds.csv"]
print("Cached artifacts for notebooks 02+:")
for f in cache_files:
    if f.exists():
        print(f"  {f}  ({f.stat().st_size / 1e6:.2f} MB)")
print()
print("Reusable functions: src/eda_utils.py -> parse_peaks, parse_adduct, expected_precursor_mz,")
print("neutral_mass_from_adduct, build_tautomer_enumerator, conn_key, compute_structure_record,")
print("mrr_at_25, validate_submission, compute_peak_stats, spectral_entropy.")
_done()


**Final findings and recommendations for notebooks 02/03**

1. **Framing.** `observation = spectrum`, `prediction unit = molecule`, `label = conn_key`
   (tautomer-canonical InChIKey14) — train on `conn_key`, not the dataset's raw `inchikey14`
   column, since tautomer canonicalization measurably merges plain-InChIKey14 groups
   (Section 3). `conn_key`/`mrr_at_25`/`validate_submission` are unit-tested and reusable from
   `src/eda_utils.py`.
2. **Column access.** Every later notebook should go through `COLUMN_MAP`/`DERIVED_ROLES`
   (Section 1) rather than raw column names — `charge` and, per split, `spectrum_id`/
   `molecule_id`/`source` are derived, not stored.
3. **Adducts.** `parse_adduct`'s generic `[nM+-frag...]z+/-` grammar covers effectively all
   real train adducts (Section 5) — no need for a hand-maintained adduct list in notebook 02.
   Test only uses 7 adduct forms (Section 2); prioritize those.
4. **Mass tolerance.** Use the `enveda-180`-specific ppm quantiles from
   `res_05_ppm_quantiles.csv` (surfaced again in `res_10_summary.csv`'s
   `recommended_ppm_tolerance_p95_test_like_source`) for candidate-generation windows, not a
   pooled-across-sources number — the pooled number is optimistic/pessimistic depending on how
   source mix compares to test.
5. **Most test-like source.** `res_07_shift_by_source.csv`/adversarial validation/mass-coverage
   all point at the same source (`most_test_like_source` in `res_10_summary.csv`) — prioritize
   it for candidate generation, similarity search, and as the source for `test_like_holdout`.
6. **Validation regime.** Report `test_like_holdout` results as the primary CV number; use
   `group_kfold` for training/model selection and `scaffold_kfold` as the harder robustness
   check (`res_09_nn_tanimoto.csv` quantifies exactly how much harder). Load `folds.csv`/
   `folds.parquet` rather than re-deriving splits.
7. **Metric harness.** `res_09_sanity_baseline.csv`'s non-trivial (neither 0 nor 1) MRR@25 is
   the evidence the harness -- candidate windowing, dedup, truncation, reciprocal rank -- works
   end to end before any real model result is trusted.
8. **Structural ceiling.** `res_03_key_collisions.csv` bounds any approach that must emit one
   "best" structure per conn_key — a small but nonzero share of conn_keys still map to >1
   distinct canonical SMILES even after tautomer canonicalization.
9. **Peak preprocessing (notebook 03).** `res_06_peak_stats_by_source.csv` — compare the
   most-test-like source's row directly against test's own row for intensity normalization
   convention, base-peak position, and peak-count regime; the quick binned-cosine preview
   (`res_06_same_vs_diff_cosine.csv`) already shows real same-vs-different-molecule separation
   before any fancier featurization.
10. **Domain shift.** The adversarial validation AUC (`res_10_summary.csv`) quantifies how
    learnable the train/test gap is overall — a high AUC means notebook 02+ should weight
    the most-test-like source and its precursor-mass/adduct regime rather than trusting
    uniform-across-train performance.

**Scale note:** check the MODE banner printed by Section 0 / `res_10_summary.csv`'s `mode`
row. If it says `smoke`, every number above is a logic check, not a real finding — flip
`CONFIG["SMOKE_TEST"]` to `False` and rerun top to bottom (caches are mode-tagged, so this is
a full, correct recomputation, not a partial one) before trusting any of it for actual
competition decisions.
